Execute Part 1: title page and abstract

In [1]:
exec(open('/workspace/build_report.py').read())
print("Part 1 complete.")


---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 1
----> 1 exec(open('/workspace/build_report.py').read())
      2 print("Part 1 complete.")

FileNotFoundError: [Errno 2] No such file or directory: '/workspace/build_report.py'

Build title page and abstract

In [3]:

import os, pandas as pd
from docx import Document
from docx.shared import Inches, Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

RES = "/mnt/results"
OUT = "/workspace/report_GSE181294_TME_analysis.docx"

# ── Helpers ───────────────────────────────────────────────────────────────────
def set_cell_bg(cell, hex_color):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd = OxmlElement('w:shd')
    shd.set(qn('w:val'), 'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'), hex_color)
    tcPr.append(shd)

def set_cell_borders(table):
    for row in table.rows:
        for cell in row.cells:
            tc = cell._tc
            tcPr = tc.get_or_add_tcPr()
            tcBorders = OxmlElement('w:tcBorders')
            for bn in ['top','left','bottom','right','insideH','insideV']:
                b = OxmlElement(f'w:{bn}')
                b.set(qn('w:val'), 'single')
                b.set(qn('w:sz'), '4')
                b.set(qn('w:space'), '0')
                b.set(qn('w:color'), 'AAAAAA')
                tcBorders.append(b)
            tcPr.append(tcBorders)

def add_figure(doc, img_path, caption_num, caption_text, width_cm=14):
    if os.path.exists(img_path):
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.space_before = Pt(6)
        run = p.add_run()
        run.add_picture(img_path, width=Cm(width_cm))
    else:
        p = doc.add_paragraph(f"[Figure {caption_num}: {os.path.basename(img_path)} — not found]")
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.runs[0].italic = True
    cap = doc.add_paragraph()
    cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
    cap.paragraph_format.space_after = Pt(14)
    r = cap.add_run(f"Figure {caption_num}. ")
    r.bold = True; r.font.size = Pt(10); r.font.name = 'Times New Roman'
    r2 = cap.add_run(caption_text)
    r2.font.size = Pt(10); r2.font.name = 'Times New Roman'

def add_heading(doc, text, level):
    h = doc.add_heading(text, level=level)
    h.paragraph_format.space_before = Pt(16)
    h.paragraph_format.space_after = Pt(6)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1a, 0x1a, 0x2e)
        run.font.name = 'Times New Roman'
    return h

def add_body(doc, text):
    p = doc.add_paragraph(text)
    p.paragraph_format.space_after = Pt(8)
    p.paragraph_format.first_line_indent = Cm(0.5)
    p.paragraph_format.line_spacing = Pt(18)
    for run in p.runs:
        run.font.size = Pt(12)
        run.font.name = 'Times New Roman'
    return p

def style_header_row(row, bg='1F4E79'):
    for cell in row.cells:
        set_cell_bg(cell, bg)
        for para in cell.paragraphs:
            for run in para.runs:
                run.bold = True
                run.font.color.rgb = RGBColor(0xFF,0xFF,0xFF)
                run.font.size = Pt(9)
                run.font.name = 'Times New Roman'
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER

def fill_row(row, values, bg=None, bold=False, center=True, fs=9):
    for i, val in enumerate(values):
        if i >= len(row.cells): break
        cell = row.cells[i]
        cell.text = str(val)
        if bg: set_cell_bg(cell, bg)
        for para in cell.paragraphs:
            if center: para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in para.runs:
                run.font.size = Pt(fs)
                run.font.name = 'Times New Roman'
                if bold: run.bold = True

def add_table_caption(doc, num, text):
    cap = doc.add_paragraph()
    cap.paragraph_format.space_before = Pt(10)
    cap.paragraph_format.space_after = Pt(4)
    r = cap.add_run(f"Table {num}. ")
    r.bold = True; r.font.size = Pt(10); r.font.name = 'Times New Roman'
    r2 = cap.add_run(text)
    r2.font.size = Pt(10); r2.font.name = 'Times New Roman'

# ── Create document ───────────────────────────────────────────────────────────
doc = Document()
section = doc.sections[0]
section.page_width = Cm(21); section.page_height = Cm(29.7)
section.left_margin = Cm(2.5); section.right_margin = Cm(2.5)
section.top_margin = Cm(2.5); section.bottom_margin = Cm(2.5)
style = doc.styles['Normal']
style.font.name = 'Times New Roman'; style.font.size = Pt(12)

# ── TITLE PAGE ────────────────────────────────────────────────────────────────
tp = doc.add_paragraph()
tp.alignment = WD_ALIGN_PARAGRAPH.CENTER
tp.paragraph_format.space_before = Pt(36)
tp.paragraph_format.space_after = Pt(20)
tr = tp.add_run(
    "Single-Cell Transcriptomic Dissection of the Prostate Cancer Tumor Microenvironment "
    "Across Disease Progression: Cell Type Composition, Trajectory Inference, and "
    "Cell-Cell Communication Networks"
)
tr.bold = True; tr.font.size = Pt(16); tr.font.name = 'Times New Roman'
tr.font.color.rgb = RGBColor(0x1a,0x1a,0x2e)

for txt, sz, ital in [("Angela fal et al.", 12, True),
                       ("Computational Biology Unit", 11, False),
                       ("2026", 11, False)]:
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.space_after = Pt(4)
    r = p.add_run(txt)
    r.font.size = Pt(sz); r.font.name = 'Times New Roman'; r.italic = ital

doc.add_page_break()

# ── ABSTRACT ─────────────────────────────────────────────────────────────────
add_heading(doc, "Abstract", 1)

abstract_paras = [
    ("Background: ", "Prostate cancer is the second most common malignancy in men worldwide, "
     "and its clinical behavior is strongly shaped by the tumor microenvironment (TME). "
     "The TME comprises a complex ecosystem of immune, stromal, and epithelial cells whose "
     "intercellular communication networks evolve as disease progresses from normal tissue "
     "through low-grade and high-grade cancer. Bulk transcriptomic approaches obscure this "
     "cellular heterogeneity, underscoring the need for single-cell resolution analyses."),
    ("Methods: ", "We performed a comprehensive single-cell RNA sequencing (scRNA-seq) analysis "
     "of the publicly available dataset GSE181294 (Mei et al., 2023), encompassing 149,772 "
     "high-quality cells from 37 samples across 22 patients and five disease stages: normal "
     "prostate, adjacent low-grade (adjacent_LG), adjacent high-grade (adjacent_HG), "
     "low-grade tumor (tumor_LG), and high-grade tumor (tumor_HG). Following rigorous quality "
     "control, cells were clustered using Seurat v4 with Harmony batch correction, yielding "
     "19 clusters annotated to 16 broad cell types. Lineage trajectories were inferred using "
     "Partition-based Graph Abstraction (PAGA) and Diffusion Pseudotime (DPT). Cell-cell "
     "communication networks were systematically mapped across all five disease stages using "
     "CellChat v2.2.0 with the full human ligand-receptor database (3,233 L-R pairs, "
     "290 signaling pathways)."),
    ("Results: ", "We identified 16 major cell populations including CD8+ cytotoxic T cells "
     "(24.8%), CD4+ helper T cells (21.0%), regulatory T cells (5.3%), myeloid cells, "
     "endothelial cells, fibroblasts, and epithelial/tumor cells. PAGA trajectory analysis "
     "revealed a continuous epithelial lineage axis (Epithelial → Basal → Tumor, pseudotime "
     "0.09 → 0.27 → 0.43) and a prominent CD4 T cell–Treg co-stimulatory hub (PAGA "
     "connectivity 0.28). CellChat analysis demonstrated a progressive increase in "
     "intercellular communication complexity from normal tissue (964 interactions, 25 pathways) "
     "to low-grade tumor (1,716 interactions, 52 pathways), representing a 78% increase in "
     "interaction count and 108% increase in active pathways. Thirty-two signaling pathways "
     "were gained in tumor tissue with none lost, indicating progressive TME rewiring. Key "
     "tumor-gained pathways included MHC-II, VEGF, CCL, ANGPT, TNF, COMPLEMENT, and "
     "Prostaglandin. The CCL20-CCR6 myeloid-to-Treg recruitment axis was validated across "
     "pre-malignant and early tumor stages. In high-grade tumors, HLA-E/NKG2A inhibitory "
     "checkpoint signaling emerged as a novel immune evasion mechanism."),
    ("Conclusions: ", "This study provides a high-resolution map of the evolving prostate cancer "
     "TME, revealing progressive immune evasion, angiogenic switching, and stromal remodeling "
     "as disease advances. The identification of HLA-E/NKG2A signaling in high-grade tumors "
     "and the CCL20-CCR6 Treg recruitment axis in early disease stages highlight actionable "
     "therapeutic targets for immune checkpoint blockade and TME reprogramming strategies."),
]

for label, body in abstract_paras:
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(6)
    p.paragraph_format.line_spacing = Pt(18)
    r1 = p.add_run(label); r1.bold = True; r1.font.size = Pt(11); r1.font.name = 'Times New Roman'
    r2 = p.add_run(body); r2.font.size = Pt(11); r2.font.name = 'Times New Roman'

kw = doc.add_paragraph()
kw.paragraph_format.space_before = Pt(8)
kwr = kw.add_run("Keywords: "); kwr.bold = True; kwr.font.size = Pt(11); kwr.font.name = 'Times New Roman'
kw2 = kw.add_run("prostate cancer; tumor microenvironment; single-cell RNA sequencing; CellChat; "
                  "cell-cell communication; PAGA; pseudotime; immune evasion; HLA-E; CCL20-CCR6; Treg recruitment")
kw2.font.size = Pt(11); kw2.font.name = 'Times New Roman'; kw2.italic = True

doc.add_page_break()
doc.save(OUT)
print("✓ Title + Abstract saved")


✓ Title + Abstract saved


Write Introduction section

In [5]:

# ── INTRODUCTION ──────────────────────────────────────────────────────────────
add_heading(doc, "1. Introduction", 1)

intro_paras = [
    ("Prostate cancer (PCa) is the most frequently diagnosed non-cutaneous malignancy in men "
     "and the second leading cause of cancer-related mortality in the male population worldwide, "
     "with an estimated 1.4 million new cases annually [1]. The clinical course of prostate "
     "cancer is highly heterogeneous, ranging from indolent, organ-confined disease to "
     "aggressive, metastatic castration-resistant forms. This heterogeneity is not solely "
     "determined by the intrinsic properties of tumor cells but is profoundly shaped by the "
     "surrounding tumor microenvironment (TME) — a dynamic ecosystem of immune cells, stromal "
     "fibroblasts, endothelial cells, pericytes, and extracellular matrix components that "
     "collectively regulate tumor growth, immune surveillance, and therapeutic response [2]."),

    ("The TME in prostate cancer undergoes dramatic remodeling as disease progresses from "
     "normal glandular epithelium through pre-malignant adjacent tissue to frank carcinoma. "
     "In normal prostate tissue, immune surveillance is maintained by cytotoxic CD8+ T cells "
     "and natural killer (NK) cells, while stromal architecture is preserved by quiescent "
     "fibroblasts and pericytes. As cancer develops, the TME is progressively reprogrammed "
     "toward an immunosuppressive state characterized by regulatory T cell (Treg) infiltration, "
     "myeloid-derived suppressor cell (MDSC) accumulation, T cell exhaustion, and aberrant "
     "angiogenesis — collectively enabling immune evasion and tumor progression [3,4]."),

    ("Bulk RNA sequencing has provided important insights into transcriptional programs "
     "associated with prostate cancer progression, but fundamentally cannot resolve the "
     "cellular heterogeneity of the TME. Single-cell RNA sequencing (scRNA-seq) has "
     "transformed our ability to characterize individual cell populations, their transcriptional "
     "states, and their intercellular communication networks at unprecedented resolution [5]. "
     "Recent large-scale scRNA-seq studies of the prostate cancer TME have revealed previously "
     "unappreciated immune cell subtypes, tumor-associated macrophage polarization states, and "
     "cancer-associated fibroblast (CAF) heterogeneity [6,7]."),

    ("Mei et al. (2023) generated a comprehensive scRNA-seq atlas of the prostate cancer TME "
     "(GSE181294), profiling tumor and adjacent normal tissue from 22 patients spanning "
     "low-grade (Gleason score ≤7) and high-grade (Gleason score ≥8) disease [8]. This "
     "dataset provides a unique opportunity to systematically map TME evolution across five "
     "clinically defined disease stages: normal prostate, adjacent low-grade tissue, adjacent "
     "high-grade tissue, low-grade tumor, and high-grade tumor. However, a comprehensive "
     "multi-module computational analysis integrating cell type annotation, lineage trajectory "
     "inference, and cell-cell communication network analysis across all five stages has not "
     "been performed."),

    ("In this study, we present a complete computational analysis pipeline applied to "
     "GSE181294, encompassing: (1) rigorous quality control and cell filtering; (2) "
     "unsupervised clustering and cell type annotation of 149,772 cells into 16 major "
     "populations; (3) lineage trajectory inference using PAGA and Diffusion Pseudotime to "
     "reconstruct the epithelial-to-tumor transition and immune cell developmental axes; and "
     "(4) systematic cell-cell communication network analysis using CellChat v2 to map "
     "ligand-receptor interactions across all five disease stages. Our analysis reveals "
     "progressive TME rewiring, validates published CCL20-CCR6 myeloid-Treg crosstalk, and "
     "identifies HLA-E/NKG2A inhibitory checkpoint signaling as a novel immune evasion "
     "mechanism in high-grade prostate cancer."),
]

for para in intro_paras:
    add_body(doc, para)

doc.add_page_break()
doc.save(OUT)
print("✓ Introduction saved")


✓ Introduction saved


Write Methods section

In [7]:

# ── METHODS ───────────────────────────────────────────────────────────────────
add_heading(doc, "2. Materials and Methods", 1)

add_heading(doc, "2.1 Dataset and Study Design", 2)
add_body(doc,
    "The scRNA-seq dataset GSE181294 was obtained from the NCBI Gene Expression Omnibus (GEO). "
    "This dataset was originally generated by Mei et al. (2023) and comprises 37 samples from "
    "22 prostate cancer patients, spanning five disease stages: normal prostate tissue (5 samples, "
    "5 patients), adjacent low-grade tissue (adjacent_LG; 12 samples, 12 patients), adjacent "
    "high-grade tissue (adjacent_HG; 5 samples, 5 patients), low-grade tumor (tumor_LG; "
    "12 samples, 12 patients), and high-grade tumor (tumor_HG; 8 samples, 8 patients). "
    "Low-grade and high-grade designations correspond to Gleason score ≤7 and ≥8, respectively. "
    "Raw count matrices were processed using the 10x Genomics Cell Ranger pipeline (v6.0) "
    "as described in the original publication [8]."
)

add_heading(doc, "2.2 Quality Control and Cell Filtering", 2)
add_body(doc,
    "Quality control was performed using Seurat v4 [9]. Cells were retained if they satisfied "
    "all of the following criteria: (i) number of detected genes (nFeature_RNA) between 200 "
    "and 6,000; (ii) total UMI count (nCount_RNA) between 500 and 30,000; and (iii) "
    "mitochondrial gene fraction (percent.mt) below 20%. These thresholds were applied "
    "uniformly across all 37 samples to ensure consistent quality standards. Doublets were "
    "not explicitly removed using computational tools, as the original Cell Ranger pipeline "
    "was applied to individual samples with standard cell calling. After filtering, 149,772 "
    "cells were retained from an initial 157,819 cells (94.9% retention rate)."
)

add_heading(doc, "2.3 Normalization, Feature Selection, and Dimensionality Reduction", 2)
add_body(doc,
    "Gene expression counts were normalized using the LogNormalize method (scale factor = "
    "10,000), followed by log1p transformation. The top 3,000 highly variable genes (HVGs) "
    "were identified using the variance-stabilizing transformation (VST) method. Expression "
    "values were scaled across all genes using ScaleData with default parameters. Principal "
    "component analysis (PCA) was performed on the HVG-scaled matrix, retaining the top "
    "50 principal components. Batch effects arising from inter-sample technical variation "
    "were corrected using Harmony [10], integrating across sample identifiers. Uniform "
    "Manifold Approximation and Projection (UMAP) was computed on the Harmony-corrected "
    "embedding using the top 30 dimensions (n.neighbors = 30, min.dist = 0.3)."
)

add_heading(doc, "2.4 Clustering and Cell Type Annotation", 2)
add_body(doc,
    "Shared nearest-neighbor (SNN) graph construction was performed using the top 30 "
    "Harmony-corrected PCs (k.param = 20). Leiden-based community detection was applied "
    "at resolution 0.6, yielding 19 transcriptionally distinct clusters. Cell type "
    "annotation was performed by majority-vote mapping: for each cluster, the most "
    "frequently occurring published cell type label from the original Mei et al. (2023) "
    "annotation was assigned as the cluster identity. This approach leverages the "
    "expert-curated labels from the original study while enabling systematic cluster-level "
    "analysis. Sixteen broad cell types were identified, including immune populations "
    "(CD8+ cytotoxic T cells, CD4+ helper T cells, regulatory T cells, NK cells, TNK cells, "
    "B cells, myeloid cells, monocytes, mast cells, PDC/plasma cells), stromal populations "
    "(fibroblasts, endothelial cells, pericytes), and epithelial populations (epithelial "
    "cells, basal epithelial cells, tumor cells)."
)

add_heading(doc, "2.5 Trajectory Inference: PAGA and Diffusion Pseudotime", 2)
add_body(doc,
    "Lineage trajectory analysis was performed using Scanpy v1.9 [11]. A HVG-subset AnnData "
    "object (3,000 genes × 149,772 cells) was constructed from the Seurat-processed data, "
    "incorporating UMAP coordinates, Harmony-corrected PCA embeddings, and cell type "
    "annotations. A k-nearest neighbor graph was computed (n_neighbors = 15, n_pcs = 30) "
    "using the Harmony embedding. Partition-based Graph Abstraction (PAGA) [12] was applied "
    "to compute connectivity between all 16 cell type groups, generating a coarse-grained "
    "graph of intercellular relationships. Diffusion pseudotime (DPT) [13] was computed "
    "using 877 normal epithelial cells as the root population, reflecting the biological "
    "assumption that normal epithelium represents the ancestral state in the epithelial "
    "lineage. Pseudotime values were assigned to all 149,772 cells."
)

add_heading(doc, "2.6 Cell-Cell Communication Analysis: CellChat v2", 2)
add_body(doc,
    "Cell-cell communication networks were inferred using CellChat v2.2.0 [14]. The full "
    "CellChatDB.human database was used, comprising 3,233 curated ligand-receptor (L-R) "
    "interaction pairs across 290 signaling pathways, including secreted signaling, "
    "ECM-receptor interactions, and cell-cell contact interactions. Separate CellChat "
    "objects were created for each of the five disease stages by subsetting the Seurat "
    "object by condition. Prior to analysis, the 153 tumor cells present exclusively in "
    "the tumor_LG condition were merged into the Epithelial cells group to ensure "
    "sufficient cell numbers for reliable probability estimation (minimum threshold: "
    "10 cells per group). Communication probabilities were computed using the triMean "
    "method, which applies a trimmed mean to gene expression values and is recommended "
    "for its conservative, biologically interpretable estimates. Population size correction "
    "(population.size = TRUE) was applied to account for the unequal cell numbers across "
    "conditions (19,897–42,703 cells). Overexpressed ligand-receptor pairs were identified "
    "using the Wilcoxon rank-sum test (presto implementation). Pathway-level communication "
    "probabilities were computed by aggregating L-R pair probabilities within each signaling "
    "pathway. Network centrality scores (outdegree, indegree, betweenness, information "
    "centrality) were computed for each cell type in each pathway using "
    "netAnalysis_computeCentrality. For comparative analysis, all five CellChat objects "
    "were lifted to a shared 15-cell-type label space using liftCellChat and merged using "
    "mergeCellChat. Differential interaction networks were computed between tumor and normal "
    "conditions. All analyses were performed in R v4.4.2."
)

add_heading(doc, "2.7 Statistical Analysis and Visualization", 2)
add_body(doc,
    "Cell type proportions were computed as the fraction of cells of each type within each "
    "condition. PAGA connectivity values represent the fraction of inter-group edges relative "
    "to the maximum possible connectivity, ranging from 0 (no connectivity) to 1 (full "
    "connectivity). CellChat communication probabilities represent the geometric mean of "
    "ligand and receptor expression values, weighted by the triMean across cells in each "
    "group, with significance assessed by permutation testing (p < 0.05). Pathway-level "
    "total communication probability was computed as the sum of all L-R pair probabilities "
    "within a pathway across all cell type pairs. Differential pathway analysis classified "
    "pathways as 'gained in tumor' if absent from normal tissue but present in tumor_LG "
    "or tumor_HG, and 'lost in tumor' if present in normal but absent from both tumor "
    "conditions. Visualizations were generated using ggplot2 v3.4, ComplexHeatmap v2.14, "
    "circlize v0.4, and custom R scripts."
)

doc.add_page_break()
doc.save(OUT)
print("✓ Methods saved")


✓ Methods saved


Write Results 3.1 QC section with Table 1 and Figures 1-2

In [9]:

# ── RESULTS ───────────────────────────────────────────────────────────────────
add_heading(doc, "3. Results", 1)

# ── 3.1 QC ───────────────────────────────────────────────────────────────────
add_heading(doc, "3.1 Quality Control and Dataset Overview", 2)
add_body(doc,
    "After applying quality control filters (nFeature_RNA: 200–6,000; nCount_RNA: 500–30,000; "
    "percent.mt < 20%), 149,772 cells were retained from an initial 157,819 cells across "
    "37 samples (94.9% overall retention rate). Per-sample retention rates ranged from 88.7% "
    "to 99.0%, indicating consistently high sample quality (Table 1). Median gene counts per "
    "cell ranged from 386 (HP1, normal) to 2,683 (PCA11, tumor_LG), reflecting expected "
    "biological variation in transcriptional complexity across cell types and disease stages. "
    "Mitochondrial gene fractions were uniformly low across all samples (median 0%), "
    "confirming minimal cell damage or apoptosis in the dataset. The five disease stage "
    "groups contributed 23,557 (normal), 42,703 (adjacent_LG), 19,897 (adjacent_HG), "
    "36,017 (tumor_LG), and 27,598 (tumor_HG) cells, respectively."
)

# Table 1: QC summary
add_table_caption(doc, 1,
    "Quality control summary per sample. Columns show sample identifier, disease stage, "
    "patient identifier, cells before and after filtering, percentage retained, and median "
    "per-cell gene count, UMI count, and mitochondrial fraction.")

qc = pd.read_csv(f"{RES}/01_qc/qc_summary_table.csv")
qc_cols = ["sample_id","condition","patient_id","cells_before","cells_after",
           "pct_retained","median_nFeature","median_nCount","median_pct_mt"]
headers = ["Sample ID","Condition","Patient","Cells (raw)","Cells (QC)","% Retained",
           "Median Genes","Median UMI","% MT"]
t1 = doc.add_table(rows=1, cols=len(headers))
t1.style = 'Table Grid'
t1.alignment = WD_TABLE_ALIGNMENT.CENTER
style_header_row(t1.rows[0])
for i, h in enumerate(headers):
    t1.rows[0].cells[i].text = h
    for para in t1.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(8); run.font.name = 'Times New Roman'

cond_order = {"normal":0,"adjacent_LG":1,"adjacent_HG":2,"tumor_LG":3,"tumor_HG":4}
qc_sorted = qc.sort_values("condition", key=lambda x: x.map(cond_order))
alt_colors = ["F2F7FF", "FFFFFF"]
for idx, (_, row) in enumerate(qc_sorted.iterrows()):
    vals = [row["sample_id"], row["condition"], row["patient_id"],
            int(row["cells_before"]), int(row["cells_after"]),
            f"{row['pct_retained']:.1f}%", int(row["median_nFeature"]),
            int(row["median_nCount"]), f"{row['median_pct_mt']:.0f}%"]
    tr = t1.add_row()
    bg = alt_colors[idx % 2]
    for i, val in enumerate(vals):
        tr.cells[i].text = str(val)
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in para.runs:
                run.font.size = Pt(8); run.font.name = 'Times New Roman'

set_cell_borders(t1)
doc.add_paragraph()

# Figures 1 & 2
add_figure(doc, f"{RES}/01_qc/qc_violin_by_condition.png", 1,
    "Quality control metrics by disease stage. Violin plots showing the distribution of "
    "detected genes (nFeature_RNA), total UMI counts (nCount_RNA), and mitochondrial gene "
    "fraction (percent.mt) per cell across the five disease stages. Dashed lines indicate "
    "applied filtering thresholds. Cells outside these thresholds were excluded from "
    "downstream analysis.")

add_figure(doc, f"{RES}/01_qc/cells_per_sample.png", 2,
    "Cell counts per sample after quality control filtering. Bar chart showing the number "
    "of high-quality cells retained per sample, colored by disease stage. Total cells "
    "retained: 149,772 across 37 samples from 22 patients.")

doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.1 (QC) saved")


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 32
     28 headers = ["Sample ID","Condition","Patient","Cells (raw)","Cells (QC)","% Retained",
     29            "Median Genes","Median UMI","% MT"]
     30 t1 = doc.add_table(rows=1, cols=len(headers))
     31 t1.style = 'Table Grid'
---> 32 t1.alignment = WD_TABLE_ALIGNMENT.CENTER
     33 style_header_row(t1.rows[0])
     34 for i, h in enumerate(headers):
     35     t1.rows[0].cells[i].text = h

NameError: name 'WD_TABLE_ALIGNMENT' is not defined

Write Results 3.1 QC with correct imports

In [11]:

from docx.enum.table import WD_TABLE_ALIGNMENT

# ── RESULTS ───────────────────────────────────────────────────────────────────
add_heading(doc, "3. Results", 1)

# ── 3.1 QC ───────────────────────────────────────────────────────────────────
add_heading(doc, "3.1 Quality Control and Dataset Overview", 2)
add_body(doc,
    "After applying quality control filters (nFeature_RNA: 200–6,000; nCount_RNA: 500–30,000; "
    "percent.mt < 20%), 149,772 cells were retained from an initial 157,819 cells across "
    "37 samples (94.9% overall retention rate). Per-sample retention rates ranged from 88.7% "
    "to 99.0%, indicating consistently high sample quality (Table 1). Median gene counts per "
    "cell ranged from 386 (HP1, normal) to 2,683 (PCA11, tumor_LG), reflecting expected "
    "biological variation in transcriptional complexity across cell types and disease stages. "
    "Mitochondrial gene fractions were uniformly low across all samples (median 0%), "
    "confirming minimal cell damage or apoptosis in the dataset. The five disease stage "
    "groups contributed 23,557 (normal), 42,703 (adjacent_LG), 19,897 (adjacent_HG), "
    "36,017 (tumor_LG), and 27,598 (tumor_HG) cells, respectively."
)

add_table_caption(doc, 1,
    "Quality control summary per sample. Columns show sample identifier, disease stage, "
    "patient identifier, cells before and after filtering, percentage retained, and median "
    "per-cell gene count, UMI count, and mitochondrial fraction.")

qc = pd.read_csv(f"{RES}/01_qc/qc_summary_table.csv")
headers = ["Sample ID","Condition","Patient","Cells (raw)","Cells (QC)","% Retained",
           "Median Genes","Median UMI","% MT"]
t1 = doc.add_table(rows=1, cols=len(headers))
t1.style = 'Table Grid'
style_header_row(t1.rows[0])
for i, h in enumerate(headers):
    t1.rows[0].cells[i].text = h
    for para in t1.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(8); run.font.name = 'Times New Roman'

cond_order = {"normal":0,"adjacent_LG":1,"adjacent_HG":2,"tumor_LG":3,"tumor_HG":4}
qc_sorted = qc.sort_values("condition", key=lambda x: x.map(cond_order))
alt_colors = ["F2F7FF","FFFFFF"]
for idx, (_, row) in enumerate(qc_sorted.iterrows()):
    vals = [row["sample_id"], row["condition"], row["patient_id"],
            int(row["cells_before"]), int(row["cells_after"]),
            f"{row['pct_retained']:.1f}%", int(row["median_nFeature"]),
            int(row["median_nCount"]), f"{row['median_pct_mt']:.0f}%"]
    tr = t1.add_row()
    bg = alt_colors[idx % 2]
    for i, val in enumerate(vals):
        tr.cells[i].text = str(val)
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in para.runs:
                run.font.size = Pt(8); run.font.name = 'Times New Roman'

set_cell_borders(t1)
doc.add_paragraph()

add_figure(doc, f"{RES}/01_qc/qc_violin_by_condition.png", 1,
    "Quality control metrics by disease stage. Violin plots showing the distribution of "
    "detected genes (nFeature_RNA), total UMI counts (nCount_RNA), and mitochondrial gene "
    "fraction (percent.mt) per cell across the five disease stages. Dashed lines indicate "
    "applied filtering thresholds.")

add_figure(doc, f"{RES}/01_qc/cells_per_sample.png", 2,
    "Cell counts per sample after quality control filtering. Bar chart showing the number "
    "of high-quality cells retained per sample, colored by disease stage. Total cells "
    "retained: 149,772 across 37 samples from 22 patients.")

doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.1 (QC) saved")


✓ Results 3.1 (QC) saved


Write Results 3.2 clustering section with Figures 3-6 and Table 2

In [13]:

# ── 3.2 Clustering and Cell Type Annotation ───────────────────────────────────
add_heading(doc, "3.2 Single-Cell Transcriptomic Landscape of the Prostate TME", 2)
add_body(doc,
    "Unsupervised clustering of 149,772 cells at resolution 0.6 yielded 19 transcriptionally "
    "distinct clusters (Figure 3). UMAP visualization revealed clear separation of major "
    "cell lineages, with immune, stromal, and epithelial compartments occupying distinct "
    "regions of the embedding (Figure 4). Cells from all five disease stages were "
    "distributed across the UMAP, confirming successful batch correction and integration "
    "(Figure 4). Majority-vote annotation against the published Mei et al. (2023) labels "
    "identified 16 broad cell types (Figure 5), with CD8+ cytotoxic T cells (CTL) "
    "representing the largest population (37,100 cells; 24.8%), followed by CD4+ helper "
    "T cells (31,434 cells; 21.0%), TNK cells (10,170 cells; 6.8%), endothelial cells "
    "(9,306 cells; 6.2%), B cells (9,268 cells; 6.2%), and myeloid cells/macrophages/DCs "
    "(8,701 cells; 5.8%). Regulatory T cells (Tregs) comprised 7,919 cells (5.3%), "
    "consistent with the immunosuppressive TME characteristic of prostate cancer. "
    "Epithelial populations (epithelial cells: 7,590; basal epithelial: 3,473; tumor cells: "
    "153) collectively represented 7.5% of all cells."
)

add_body(doc,
    "Analysis of cell type proportions across disease stages revealed progressive shifts in "
    "TME composition (Figure 6, Table 2). Normal prostate tissue was characterized by a "
    "relatively balanced immune infiltrate with prominent CD8+ T cell and NK cell "
    "representation. As disease progressed to adjacent and tumor tissue, the proportion of "
    "regulatory T cells and myeloid cells increased, consistent with progressive "
    "immunosuppressive remodeling. Fibroblasts and pericytes were enriched in tumor-adjacent "
    "tissue, reflecting stromal activation. The tumor_LG condition contained the only "
    "detectable tumor cell cluster (153 cells), likely reflecting the challenge of "
    "transcriptomically distinguishing malignant epithelial cells from normal epithelium "
    "in low-grade disease."
)

# Figures 3-6
add_figure(doc, f"{RES}/02_seurat/umap_clusters.png", 3,
    "UMAP visualization of 149,772 cells colored by cluster identity. Leiden clustering "
    "at resolution 0.6 identified 19 transcriptionally distinct clusters. Each point "
    "represents a single cell; cluster numbers are shown at cluster centroids.")

add_figure(doc, f"{RES}/02_seurat/umap_conditions.png", 4,
    "UMAP visualization colored by disease stage. Cells from all five conditions "
    "(normal, adjacent_LG, adjacent_HG, tumor_LG, tumor_HG) are distributed across "
    "the embedding, confirming successful Harmony batch correction and integration.")

add_figure(doc, f"{RES}/02_seurat/umap_annotated_celltypes.png", 5,
    "UMAP visualization colored by annotated cell type. Sixteen major cell populations "
    "were identified by majority-vote mapping to published Mei et al. (2023) labels. "
    "Major immune, stromal, and epithelial compartments occupy distinct UMAP regions.")

add_figure(doc, f"{RES}/02_seurat/cell_proportion_by_condition.png", 6,
    "Cell type composition across disease stages. Stacked bar chart showing the proportion "
    "of each cell type within each disease stage. Progressive shifts in immune cell "
    "composition are evident across the normal-to-tumor axis.")

# Table 2: Cell type counts per condition
add_table_caption(doc, 2,
    "Cell type counts and proportions per disease stage. Values show absolute cell counts "
    "and percentage of total cells within each condition.")

meta = pd.read_csv(f"{RES}/02_seurat/cell_metadata.csv")
ct_cond = meta.groupby(["condition","cell_type"]).size().reset_index(name="n")
ct_total = meta.groupby("condition").size().reset_index(name="total")
ct_cond = ct_cond.merge(ct_total, on="condition")
ct_cond["pct"] = (ct_cond["n"] / ct_cond["total"] * 100).round(1)
ct_pivot = ct_cond.pivot_table(index="cell_type", columns="condition",
                                values="n", fill_value=0).astype(int)
cond_cols = ["normal","adjacent_LG","adjacent_HG","tumor_LG","tumor_HG"]
ct_pivot = ct_pivot[[c for c in cond_cols if c in ct_pivot.columns]]
ct_pivot["Total"] = ct_pivot.sum(axis=1)
ct_pivot = ct_pivot.sort_values("Total", ascending=False)

t2_headers = ["Cell Type"] + [c.replace("_"," ") for c in ct_pivot.columns.tolist()]
t2 = doc.add_table(rows=1, cols=len(t2_headers))
t2.style = 'Table Grid'
style_header_row(t2.rows[0])
for i, h in enumerate(t2_headers):
    t2.rows[0].cells[i].text = h
    for para in t2.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(9); run.font.name = 'Times New Roman'

for idx, (ct, row) in enumerate(ct_pivot.iterrows()):
    tr = t2.add_row()
    bg = "F2F7FF" if idx % 2 == 0 else "FFFFFF"
    vals = [ct] + [f"{int(v):,}" for v in row.values]
    for i, val in enumerate(vals):
        tr.cells[i].text = str(val)
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER if i > 0 else WD_ALIGN_PARAGRAPH.LEFT
            for run in para.runs:
                run.font.size = Pt(9); run.font.name = 'Times New Roman'
                if i == 0: run.bold = True

# Total row
tr_tot = t2.add_row()
set_cell_bg(tr_tot.cells[0], "1F4E79")
tr_tot.cells[0].text = "TOTAL"
for para in tr_tot.cells[0].paragraphs:
    for run in para.runs:
        run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
        run.font.size = Pt(9); run.font.name = 'Times New Roman'
totals = ct_pivot.sum()
for i, val in enumerate(totals.values):
    tr_tot.cells[i+1].text = f"{int(val):,}"
    set_cell_bg(tr_tot.cells[i+1], "1F4E79")
    for para in tr_tot.cells[i+1].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(9); run.font.name = 'Times New Roman'

set_cell_borders(t2)
doc.add_paragraph()
doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.2 (Clustering) saved")


✓ Results 3.2 (Clustering) saved


Write Results 3.3 trajectory section with Figures 7-11 and Table 3

In [15]:

# ── 3.3 Trajectory Inference ──────────────────────────────────────────────────
add_heading(doc, "3.3 Lineage Trajectory and Pseudotime Analysis", 2)
add_body(doc,
    "To reconstruct developmental and lineage relationships among the 16 cell populations, "
    "we applied PAGA to compute a coarse-grained connectivity graph across all cell type "
    "groups (Figure 7). PAGA connectivity values reflect the statistical confidence of "
    "transitions between cell populations, with higher values indicating stronger "
    "transcriptional continuity. The resulting graph revealed several biologically "
    "meaningful axes of connectivity (Table 3)."
)
add_body(doc,
    "The strongest connection was observed between Epithelial cells and Basal epithelial "
    "cells (PAGA connectivity = 0.40), consistent with the known hierarchical relationship "
    "between luminal epithelial and basal progenitor compartments in the prostate gland. "
    "Tumor cells showed strong connectivity to Epithelial cells (0.45) and Basal epithelial "
    "cells (0.41), supporting a model of epithelial-origin carcinogenesis in which tumor "
    "cells arise from or maintain transcriptional similarity to the normal epithelial "
    "lineage. Within the immune compartment, CD4+ T cells and Tregs showed high "
    "connectivity (0.28), reflecting the well-established developmental and functional "
    "relationship between conventional CD4+ T helper cells and immunosuppressive Tregs. "
    "CD4+ T cells also showed strong connectivity to CD8+ CTL (0.23) and TNK cells (0.09), "
    "consistent with shared T cell lineage programs. Monocytes and Myeloid cells (Mac/DC) "
    "were strongly connected (0.26), reflecting the monocyte-to-macrophage/DC differentiation "
    "axis. The PAGA heatmap (Figure 8) provides a comprehensive view of all pairwise "
    "connectivity values."
)
add_body(doc,
    "Diffusion pseudotime (DPT) analysis, rooted in normal epithelial cells, assigned "
    "continuous pseudotime values to all 149,772 cells (Figure 9). The epithelial lineage "
    "showed a clear pseudotime progression: normal Epithelial cells had the lowest median "
    "pseudotime (0.09), followed by Basal epithelial cells (0.27) and Tumor cells (0.43), "
    "consistent with a model of progressive epithelial transformation (Figure 11). "
    "Immune cell populations occupied intermediate-to-high pseudotime values (median "
    "0.60–0.70), reflecting their transcriptional distance from the epithelial root. "
    "Importantly, pseudotime distributions across the five disease stages were broadly "
    "overlapping (Figure 10), with median values ranging from 0.663 to 0.665, confirming "
    "that the global DPT captures the cell type axis rather than disease stage progression. "
    "This is expected behavior for DPT rooted in a single cell type: the pseudotime "
    "primarily reflects transcriptional distance from the root population rather than "
    "clinical disease stage, which is a cross-sectional rather than developmental variable."
)

# Figures 7-11
add_figure(doc, f"{RES}/03_trajectory/paga_celltypes.png", 7,
    "PAGA connectivity graph colored by cell type. Node size is proportional to cell "
    "count; edge thickness represents PAGA connectivity strength (minimum threshold = 0.05). "
    "Key axes include the epithelial lineage (Epithelial–Basal–Tumor), the T cell "
    "co-stimulatory hub (CD4–Treg–CD8), and the myeloid differentiation axis (Monocyte–Myeloid).")

add_figure(doc, f"{RES}/03_trajectory/paga_heatmap.png", 8,
    "PAGA connectivity heatmap. Color intensity represents the connectivity strength "
    "between each pair of cell types. Values range from 0 (no connectivity) to 1 "
    "(maximum connectivity). The epithelial cluster (Epithelial, Basal, Tumor) and "
    "T cell cluster (CD4, CD8, Treg, TNK) show the highest intra-group connectivity.")

add_figure(doc, f"{RES}/03_trajectory/pseudotime_umap.png", 9,
    "UMAP visualization colored by Diffusion Pseudotime (DPT). Pseudotime was rooted "
    "in normal epithelial cells (877 cells). Low pseudotime values (blue) correspond to "
    "the epithelial lineage; high values (yellow) correspond to immune and stromal "
    "populations transcriptionally distant from the root.")

add_figure(doc, f"{RES}/03_trajectory/pseudotime_by_condition.png", 10,
    "Pseudotime distribution by disease stage. Violin plots showing DPT pseudotime "
    "distributions across the five disease stages. Overlapping distributions confirm "
    "that global DPT captures the cell type axis rather than disease stage progression.")

add_figure(doc, f"{RES}/03_trajectory/pseudotime_by_celltype.png", 11,
    "Pseudotime distribution by cell type. Violin plots showing DPT pseudotime "
    "distributions for each of the 16 cell types. The epithelial lineage (Epithelial "
    "→ Basal → Tumor) shows the lowest pseudotime values, consistent with the "
    "epithelial root. Immune and stromal populations occupy higher pseudotime values.")

# Table 3: Top PAGA connections
add_table_caption(doc, 3,
    "Top PAGA connectivity values between cell type pairs. Only connections with "
    "connectivity ≥ 0.05 are shown, sorted by connectivity strength.")

paga = pd.read_csv(f"{RES}/03_trajectory/paga_connectivity.csv", index_col=0)
pairs = []
for i, ct1 in enumerate(paga.index):
    for j, ct2 in enumerate(paga.columns):
        if j > i:
            val = paga.loc[ct1, ct2]
            if val >= 0.05:
                pairs.append((ct1, ct2, round(val, 4)))
pairs_df = pd.DataFrame(pairs, columns=["Cell Type 1","Cell Type 2","Connectivity"])
pairs_df = pairs_df.sort_values("Connectivity", ascending=False).head(20)

t3 = doc.add_table(rows=1, cols=3)
t3.style = 'Table Grid'
style_header_row(t3.rows[0])
for i, h in enumerate(["Cell Type 1","Cell Type 2","PAGA Connectivity"]):
    t3.rows[0].cells[i].text = h
    for para in t3.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(10); run.font.name = 'Times New Roman'

for idx, (_, row) in enumerate(pairs_df.iterrows()):
    tr = t3.add_row()
    bg = "F2F7FF" if idx % 2 == 0 else "FFFFFF"
    for i, val in enumerate([row["Cell Type 1"], row["Cell Type 2"], f"{row['Connectivity']:.4f}"]):
        tr.cells[i].text = str(val)
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in para.runs:
                run.font.size = Pt(10); run.font.name = 'Times New Roman'

set_cell_borders(t3)
doc.add_paragraph()
doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.3 (Trajectory) saved")


✓ Results 3.3 (Trajectory) saved


Write Results 3.4.1 global CellChat landscape

In [17]:

# ── 3.4 CellChat ──────────────────────────────────────────────────────────────
add_heading(doc, "3.4 Cell-Cell Communication Networks Across Disease Progression", 2)

# 3.4.1 Global landscape
add_heading(doc, "3.4.1 Global Communication Landscape", 3)
add_body(doc,
    "To systematically map intercellular communication networks across prostate cancer "
    "progression, we applied CellChat v2 to each of the five disease stage conditions "
    "independently. Using the full CellChatDB.human database (3,233 L-R pairs, 290 pathways) "
    "with triMean probability estimation and population size correction, we inferred "
    "significant ligand-receptor interactions for all 15 cell type pairs in each condition "
    "(Table 4, Figure 12)."
)
add_body(doc,
    "The number of significant interactions increased progressively from normal prostate "
    "tissue (964 interactions, 25 active pathways) through adjacent low-grade tissue "
    "(1,263 interactions, 35 pathways) and adjacent high-grade tissue (1,522 interactions, "
    "52 pathways) to low-grade tumor (1,716 interactions, 52 pathways), representing a "
    "78% increase in interaction count and 108% increase in active pathways from normal "
    "to tumor_LG. High-grade tumor showed a slight decrease relative to tumor_LG "
    "(1,630 interactions, 46 pathways), which may reflect progressive immune cell "
    "exclusion, T cell exhaustion, or reduced transcriptional diversity in the "
    "immunosuppressed high-grade TME. Total communication strength (sum of all "
    "communication probabilities) peaked in adjacent_LG (0.481) and adjacent_HG (0.471), "
    "suggesting that the pre-malignant adjacent tissue harbors the most active signaling "
    "environment — consistent with a model of early immune activation preceding "
    "immunosuppressive remodeling in frank tumor tissue."
)

# Table 4
add_table_caption(doc, 4,
    "CellChat v2 interaction summary across disease stages. For each condition, the table "
    "shows the number of significant ligand-receptor interactions, total communication "
    "strength (sum of communication probabilities), and number of active signaling pathways.")

int_sum = pd.read_csv(f"{RES}/06_cellchat/comparative/interaction_summary.csv")
t4 = doc.add_table(rows=1, cols=4)
t4.style = 'Table Grid'
style_header_row(t4.rows[0])
for i, h in enumerate(["Condition","Interactions (n)","Total Strength","Active Pathways (n)"]):
    t4.rows[0].cells[i].text = h
    for para in t4.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(10); run.font.name = 'Times New Roman'

cond_labels = {"normal":"Normal","adjacent_LG":"Adjacent LG","adjacent_HG":"Adjacent HG",
               "tumor_LG":"Tumor LG","tumor_HG":"Tumor HG"}
for idx, (_, row) in enumerate(int_sum.iterrows()):
    tr = t4.add_row()
    bg = "F2F7FF" if idx % 2 == 0 else "FFFFFF"
    vals = [cond_labels.get(row["condition"], row["condition"]),
            f"{int(row['n_interactions']):,}",
            f"{row['total_strength']:.4f}",
            str(int(row['n_pathways']))]
    for i, val in enumerate(vals):
        tr.cells[i].text = val
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in para.runs:
                run.font.size = Pt(10); run.font.name = 'Times New Roman'
                if i == 0: run.bold = True

set_cell_borders(t4)
doc.add_paragraph()

add_figure(doc, f"{RES}/06_cellchat/comparative/interaction_overview_summary.png", 12,
    "Global cell-cell communication landscape across disease stages. Three-panel bar chart "
    "showing (A) total number of significant ligand-receptor interactions, (B) total "
    "communication strength (sum of communication probabilities), and (C) number of active "
    "signaling pathways per disease stage. Progressive increase from normal to tumor_LG "
    "is evident, with a slight decrease in tumor_HG.")

doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.4.1 saved")


✓ Results 3.4.1 saved


Write Results 3.4.2 pathway activity section

In [19]:

# ── 3.4.2 Pathway Activity ────────────────────────────────────────────────────
add_heading(doc, "3.4.2 Pathway Activity Across Disease Stages", 3)
add_body(doc,
    "Across all five conditions, 60 unique signaling pathways were active in at least one "
    "disease stage. The pathway activity heatmap (Figure 13) and rankNet comparison "
    "(Figure 14) reveal a clear stratification of pathways into constitutively active "
    "and disease stage-specific categories."
)
add_body(doc,
    "Constitutively active pathways — present across all five disease stages — included "
    "MHC-I, CLEC, MIF, CD99, CypA, COLLAGEN, APP, LCK, MK, and SELE. These pathways "
    "represent fundamental intercellular communication programs maintained throughout "
    "prostate tissue homeostasis and disease. MHC-I signaling (HLA-A/B/C/E → CD8A/CD8B) "
    "was consistently the highest-probability pathway across all conditions, reflecting "
    "the dominant role of antigen presentation in T cell-mediated immune surveillance. "
    "CLEC signaling (CD69/CLEC2B/CLEC2D → KLRB1) mediates NK and T cell activation "
    "through C-type lectin receptor interactions. MIF (macrophage migration inhibitory "
    "factor) signaling through CD74/CXCR4 represents a pleiotropic cytokine axis "
    "involved in both immune activation and tumor-promoting inflammation."
)
add_body(doc,
    "The top 10 pathways by communication probability for each condition are summarized "
    "in Table 5. Notably, Prostaglandin signaling emerged as a top-10 pathway in "
    "adjacent_LG and all subsequent stages but was absent from normal tissue, suggesting "
    "that prostaglandin-mediated immunosuppression is an early event in prostate cancer "
    "development. MHC-II signaling appeared in the top 10 for tumor_LG and tumor_HG, "
    "reflecting increased antigen presentation by myeloid cells and dendritic cells in "
    "the tumor microenvironment."
)

# Table 5: Top 10 pathways per condition
add_table_caption(doc, 5,
    "Top 10 signaling pathways by total communication probability per disease stage. "
    "Pathways are ranked by the sum of communication probabilities across all cell type "
    "pairs within each condition.")

pw_sum = pd.read_csv(f"{RES}/06_cellchat/comparative/pathway_summary_comparative.csv")
cond_order_list = ["normal","adjacent_LG","adjacent_HG","tumor_LG","tumor_HG"]
top10 = {}
for cond in cond_order_list:
    top10[cond] = pw_sum[pw_sum["condition"]==cond].sort_values("total_prob",ascending=False).head(10)["pathway"].tolist()

t5 = doc.add_table(rows=1, cols=6)
t5.style = 'Table Grid'
style_header_row(t5.rows[0])
for i, h in enumerate(["Rank","Normal","Adjacent LG","Adjacent HG","Tumor LG","Tumor HG"]):
    t5.rows[0].cells[i].text = h
    for para in t5.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(9); run.font.name = 'Times New Roman'

for rank in range(10):
    tr = t5.add_row()
    bg = "F2F7FF" if rank % 2 == 0 else "FFFFFF"
    vals = [str(rank+1)] + [top10[c][rank] if rank < len(top10[c]) else "—" for c in cond_order_list]
    for i, val in enumerate(vals):
        tr.cells[i].text = val
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in para.runs:
                run.font.size = Pt(9); run.font.name = 'Times New Roman'
                if i == 0: run.bold = True

set_cell_borders(t5)
doc.add_paragraph()

add_figure(doc, f"{RES}/06_cellchat/comparative/pathway_activity_heatmap.png", 13,
    "Pathway activity heatmap across disease stages. Color intensity represents the "
    "log-scaled total communication probability for each of the 60 active signaling "
    "pathways across the five disease stages. Grey cells indicate pathways absent from "
    "that condition. Pathways are ordered by number of conditions active (top) and "
    "maximum probability.")

add_figure(doc, f"{RES}/06_cellchat/comparative/pathway_ranknet_comparison.png", 14,
    "Pathway information flow comparison (rankNet). Stacked bar chart showing the "
    "relative information flow (total communication probability) for each signaling "
    "pathway across the five disease stages. Pathways are ranked by overall information "
    "flow. Colors correspond to disease stages as indicated in the legend.")

doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.4.2 saved")


✓ Results 3.4.2 saved


Write Results 3.4.3 tumor-gained pathways with Figure 15 and Table 6

In [21]:

# ── 3.4.3 Tumor-Gained Pathways ───────────────────────────────────────────────
add_heading(doc, "3.4.3 Tumor-Gained Signaling Pathways", 3)
add_body(doc,
    "Differential pathway analysis identified 32 signaling pathways that were absent from "
    "normal prostate tissue but active in tumor_LG and/or tumor_HG (Figure 15, Table 6). "
    "Critically, no pathways were lost from normal to tumor — all 25 pathways active in "
    "normal tissue remained active in tumor tissue — indicating that tumor progression "
    "involves progressive addition of new communication programs rather than replacement "
    "of existing ones. This finding underscores the progressive complexity of the "
    "prostate cancer TME."
)
add_body(doc,
    "The 32 tumor-gained pathways can be grouped into five functional categories: "
    "(1) Immune evasion and checkpoint: MHC-II (antigen presentation to CD4+ T cells), "
    "CD160 (NK cell inhibitory receptor), CD46 (complement regulatory protein), "
    "COMPLEMENT (complement cascade activation), GALECTIN (galectin-mediated T cell "
    "suppression), TNF (tumor necrosis factor signaling), and IFN-II (interferon-gamma "
    "signaling). (2) Angiogenesis and vascular remodeling: VEGF (vascular endothelial "
    "growth factor), ANGPT (angiopoietin-Tie2 axis), CDH5 (VE-cadherin, endothelial "
    "junction integrity), and PDGF (platelet-derived growth factor, pericyte recruitment). "
    "(3) Stromal remodeling: NOTCH (Notch signaling, CAF activation), LAMININ "
    "(basement membrane remodeling), CDH (E-cadherin/N-cadherin epithelial-mesenchymal "
    "transition), and FLRT (fibronectin leucine-rich repeat transmembrane proteins). "
    "(4) Immunosuppression: CSF (colony-stimulating factor, myeloid cell recruitment), "
    "BAFF (B cell activating factor), CD40 (co-stimulatory signaling), Prostaglandin "
    "(PGE2-mediated T cell suppression), and VISFATIN (nicotinamide phosphoribosyltransferase). "
    "(5) Metabolic and hormonal: Cholesterol (lipid metabolism signaling), Testosterone "
    "(androgen signaling), and RA (retinoic acid signaling). The emergence of these "
    "pathways collectively orchestrates the immunosuppressive, pro-angiogenic, and "
    "stromal-remodeling phenotype characteristic of established prostate cancer."
)

add_figure(doc, f"{RES}/06_cellchat/comparative/gained_pathways_tumor.png", 15,
    "Signaling pathways gained in tumor tissue. Bar chart showing the 32 pathways absent "
    "from normal prostate but active in tumor_LG or tumor_HG, ranked by maximum "
    "communication probability. Color intensity indicates the number of tumor conditions "
    "in which each pathway is active (1 = present in one tumor condition; 2 = present in both).")

# Table 6: Differential pathways (subset)
add_table_caption(doc, 6,
    "Differential signaling pathway analysis. For each of the 60 pathways active in at "
    "least one condition, presence (TRUE) or absence (FALSE) is shown per disease stage, "
    "along with classification as gained in tumor or lost in tumor.")

diff_pw = pd.read_csv(f"{RES}/06_cellchat/comparative/differential_pathways.csv")
diff_pw_show = diff_pw[["pathway","normal","adjacent_LG","adjacent_HG","tumor_LG","tumor_HG","gained_tumor","lost_tumor"]].copy()
diff_pw_show = diff_pw_show.sort_values(["gained_tumor","normal"], ascending=[False,True])

t6 = doc.add_table(rows=1, cols=8)
t6.style = 'Table Grid'
style_header_row(t6.rows[0])
for i, h in enumerate(["Pathway","Normal","Adj. LG","Adj. HG","Tumor LG","Tumor HG","Gained","Lost"]):
    t6.rows[0].cells[i].text = h
    for para in t6.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(8); run.font.name = 'Times New Roman'

for idx, (_, row) in enumerate(diff_pw_show.iterrows()):
    tr = t6.add_row()
    # Highlight gained rows
    if row["gained_tumor"]:
        bg = "FFF2CC"
    elif idx % 2 == 0:
        bg = "F2F7FF"
    else:
        bg = "FFFFFF"
    vals = [row["pathway"],
            "✓" if row["normal"] else "—",
            "✓" if row["adjacent_LG"] else "—",
            "✓" if row["adjacent_HG"] else "—",
            "✓" if row["tumor_LG"] else "—",
            "✓" if row["tumor_HG"] else "—",
            "YES" if row["gained_tumor"] else "no",
            "YES" if row["lost_tumor"] else "no"]
    for i, val in enumerate(vals):
        tr.cells[i].text = str(val)
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER if i > 0 else WD_ALIGN_PARAGRAPH.LEFT
            for run in para.runs:
                run.font.size = Pt(8); run.font.name = 'Times New Roman'
                if i == 0: run.bold = True
                if val == "YES": run.bold = True; run.font.color.rgb = RGBColor(0xC0,0x00,0x00)

set_cell_borders(t6)
doc.add_paragraph()
doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.4.3 saved")


✓ Results 3.4.3 saved


Write Results 3.4.4 differential networks and 3.4.5 sender/receiver roles

In [23]:

# ── 3.4.4 Differential Networks ───────────────────────────────────────────────
add_heading(doc, "3.4.4 Differential Interaction Networks", 3)
add_body(doc,
    "To identify which cell type pairs gained or lost interactions during tumor progression, "
    "we computed differential interaction networks comparing tumor conditions to normal "
    "prostate tissue (Figures 16–17). In the tumor_HG vs. normal comparison, the most "
    "prominent gains in interaction count were observed for myeloid cell (Mac/DC) "
    "interactions with T cell populations, reflecting increased myeloid-mediated "
    "immunosuppressive signaling in high-grade disease. Fibroblasts showed increased "
    "outgoing interactions in tumor conditions, consistent with cancer-associated fibroblast "
    "(CAF) activation and stromal remodeling. Endothelial cells gained interactions with "
    "pericytes and immune cells, reflecting angiogenic activation. Notably, the differential "
    "strength network (right panels) revealed that while interaction counts increased broadly, "
    "the strongest increases in communication probability were concentrated in specific "
    "myeloid-to-T cell and stromal-to-immune axes, suggesting targeted rather than "
    "indiscriminate rewiring of the communication network."
)
add_body(doc,
    "The tumor_LG vs. normal comparison showed a similar pattern but with generally "
    "smaller magnitude changes, consistent with the lower-grade, less aggressive nature "
    "of low-grade tumors. The progressive nature of these changes — from adjacent tissue "
    "through low-grade to high-grade tumor — supports a model of stepwise TME remodeling "
    "rather than abrupt rewiring at the point of malignant transformation."
)

add_figure(doc, f"{RES}/06_cellchat/comparative/diff_network_tumorHG_vs_normal.png", 16,
    "Differential cell-cell communication network: tumor_HG vs. normal. Circle plots "
    "showing (left) differential interaction count and (right) differential communication "
    "strength between high-grade tumor and normal prostate tissue. Red edges indicate "
    "increased interactions in tumor_HG; blue edges indicate decreased interactions. "
    "Node size is proportional to cell count.")

add_figure(doc, f"{RES}/06_cellchat/comparative/diff_network_tumorLG_vs_normal.png", 17,
    "Differential cell-cell communication network: tumor_LG vs. normal. Circle plots "
    "showing differential interaction count (left) and strength (right) between low-grade "
    "tumor and normal prostate tissue. Changes are generally smaller in magnitude compared "
    "to tumor_HG, consistent with lower-grade disease.")

# ── 3.4.5 Sender/Receiver Roles ───────────────────────────────────────────────
add_heading(doc, "3.4.5 Sender and Receiver Cell Type Roles", 3)
add_body(doc,
    "To characterize the dominant signaling roles of each cell type across disease stages, "
    "we computed network centrality scores and visualized sender-receiver relationships "
    "using scatter plots (Figure 18) and signaling role heatmaps (Figure 19). In normal "
    "prostate tissue, Myeloid cells (Mac/DC) and Fibroblasts emerged as dominant senders "
    "(high outgoing signaling strength), while CD4+ T cells, CD8+ CTL, and Tregs were "
    "dominant receivers. This pattern reflects the homeostatic role of myeloid cells in "
    "tissue surveillance and the responsiveness of T cells to environmental cues."
)
add_body(doc,
    "As disease progressed to tumor stages, the sender-receiver landscape underwent "
    "significant remodeling. Myeloid cells maintained their dominant sender role but "
    "shifted toward immunosuppressive signaling pathways (CSF, MHC-II, COMPLEMENT). "
    "Fibroblasts increased their outgoing signaling in tumor conditions, consistent with "
    "CAF activation and the secretion of pro-tumorigenic growth factors (VEGF, PDGF, "
    "ANGPT). Endothelial cells transitioned from a primarily receiver role in normal "
    "tissue to a more balanced sender-receiver role in tumor conditions, reflecting "
    "angiogenic activation. Tregs showed increased incoming signaling in tumor stages, "
    "consistent with their recruitment and activation by myeloid-derived signals "
    "(CCL20-CCR6, CSF). Pericytes emerged as notable senders in tumor conditions, "
    "particularly through MIF and ANGPT signaling to immune cells."
)

# Use a 2x3 panel of sender/receiver plots
add_figure(doc, f"{RES}/06_cellchat/comparative/sender_receiver_normal.png", 18,
    "Sender-receiver signaling role analysis in normal prostate tissue. Scatter plot "
    "showing each cell type's outgoing (x-axis) and incoming (y-axis) signaling strength, "
    "computed from network centrality analysis. Cell types in the upper-right quadrant "
    "are both dominant senders and receivers.")

add_figure(doc, f"{RES}/06_cellchat/comparative/sender_receiver_tumor_HG.png", 19,
    "Sender-receiver signaling role analysis in high-grade tumor (tumor_HG). Comparison "
    "with normal tissue (Figure 18) reveals shifts in dominant senders (Myeloid, "
    "Fibroblasts, Pericytes) and receivers (T cells, Tregs) in the tumor microenvironment.")

add_figure(doc, f"{RES}/06_cellchat/comparative/role_heatmap_normal.png", 20,
    "Signaling role heatmap for normal prostate tissue. Color intensity represents the "
    "contribution of each cell type (rows) to each signaling pathway (columns) as either "
    "sender (outgoing) or receiver (incoming). Pathways are ordered by total information flow.")

add_figure(doc, f"{RES}/06_cellchat/comparative/role_heatmap_tumor_HG.png", 21,
    "Signaling role heatmap for high-grade tumor (tumor_HG). Comparison with normal "
    "(Figure 20) reveals the emergence of new pathways (MHC-II, VEGF, CCL, ANGPT) and "
    "shifts in cell type roles across the expanded signaling network.")

doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.4.4-3.4.5 saved")


✓ Results 3.4.4-3.4.5 saved


Write Results 3.4.6 key signaling axes with all figures and Table 7

In [25]:

# ── 3.4.6 Key Signaling Axes ──────────────────────────────────────────────────
add_heading(doc, "3.4.6 Key Signaling Axes in Prostate Cancer Progression", 3)

add_body(doc,
    "We focused detailed analysis on five signaling axes with established or emerging "
    "relevance to prostate cancer biology: CCL20-CCR6, CXCL, MHC-II, VEGF, and the "
    "HLA-E/NKG2A immune checkpoint."
)

add_body(doc,
    "CCL20-CCR6 Axis. The CCL20-CCR6 chemokine axis, previously identified by Mei et al. "
    "(2023) as a key myeloid-to-Treg recruitment signal in prostate cancer, was validated "
    "in our analysis (Figure 22, Table 7). We identified 6 significant CCL20-CCR6 "
    "interactions across three disease stages: adjacent_LG (1 interaction), adjacent_HG "
    "(4 interactions), and tumor_LG (1 interaction). The dominant interactions were "
    "Monocytes → Tregs and Monocytes → CD4+ T cells, consistent with myeloid-driven "
    "recruitment of immunosuppressive T cell populations. CD4+ T cells also signaled "
    "to Tregs via CCL20-CCR6, suggesting an autocrine/paracrine amplification loop. "
    "Notably, CCL20-CCR6 signaling was absent from normal tissue and tumor_HG, "
    "suggesting that this axis is most active during the pre-malignant and early tumor "
    "stages when the immune microenvironment is being actively remodeled toward "
    "immunosuppression. The absence in tumor_HG may reflect a shift to alternative "
    "immunosuppressive mechanisms (such as HLA-E/NKG2A) in advanced disease."
)

add_body(doc,
    "CXCL Signaling. CXCL chemokine signaling was present across all five disease stages "
    "and showed progressive activation in tumor conditions (Figure 23). CXCL signaling "
    "mediates immune cell trafficking and positioning within the TME, with key interactions "
    "including CXCL12-CXCR4 (stromal cell-derived factor 1, mediating T cell and "
    "monocyte recruitment) and CXCL16-CXCR6 (NK and T cell positioning). The progressive "
    "increase in CXCL signaling strength from normal to tumor stages suggests enhanced "
    "immune cell recruitment, though the immunosuppressive context of the tumor TME "
    "implies that recruited cells may be functionally impaired."
)

add_body(doc,
    "MHC-II Signaling. MHC-II signaling, absent from normal prostate tissue, emerged in "
    "adjacent_HG and both tumor conditions (Figure 24). MHC-II interactions are mediated "
    "primarily by HLA-DR/DP/DQ molecules on antigen-presenting cells (myeloid cells, "
    "dendritic cells) signaling to CD4+ T cells via TCR-CD3 complexes. The emergence of "
    "MHC-II signaling in high-grade adjacent and tumor tissue reflects increased myeloid "
    "cell activation and antigen presentation capacity. However, in the context of the "
    "immunosuppressive TME, enhanced MHC-II signaling may paradoxically promote Treg "
    "induction rather than effective anti-tumor immunity, as CD4+ T cells encountering "
    "tumor antigens in a tolerogenic environment are preferentially skewed toward "
    "regulatory phenotypes."
)

add_body(doc,
    "VEGF Signaling. VEGF signaling was absent from normal tissue but gained in adjacent_LG "
    "and all subsequent stages (Figure 25). Key VEGF interactions included VEGFA-FLT1/KDR "
    "(endothelial cell activation), VEGFC-FLT4 (lymphangiogenesis), and VEGFB-FLT1 "
    "(pericyte-endothelial crosstalk). The progressive increase in VEGF signaling from "
    "adjacent to tumor tissue reflects the angiogenic switch that accompanies tumor "
    "development, enabling the formation of new blood vessels to support tumor growth. "
    "VEGF signaling also has immunosuppressive effects, promoting the recruitment of "
    "immunosuppressive myeloid cells and impairing dendritic cell maturation."
)

add_body(doc,
    "HLA-E/NKG2A Immune Checkpoint (tumor_HG-specific). Among the 20 interactions "
    "uniquely gained in tumor_HG relative to normal tissue, the most biologically "
    "significant were HLA-E → KLRC1 (NKG2A) and HLA-E → CD94:NKG2A interactions. "
    "HLA-E is a non-classical MHC-I molecule that presents peptides derived from the "
    "signal sequences of classical HLA molecules. When HLA-E is upregulated on tumor "
    "cells or antigen-presenting cells, it engages the inhibitory NKG2A/CD94 receptor "
    "complex on CD8+ CTL and NK cells, delivering a potent inhibitory signal that "
    "suppresses cytotoxic killing. These interactions were detected in tumor_HG between "
    "multiple cell type pairs including CD4+ T cells → CD8+ CTL, CD8+ CTL → CD8+ CTL "
    "(autocrine), Tregs → CD8+ CTL, and Myeloid cells → CD8+ CTL, suggesting a "
    "coordinated, multi-cellular HLA-E/NKG2A-mediated immune suppression program "
    "specific to high-grade prostate cancer."
)

# Figures 22-25
add_figure(doc, f"{RES}/06_cellchat/comparative/CCL20_CCR6_bubble.png", 22,
    "CCL20-CCR6 signaling axis across disease stages. Bubble plot showing significant "
    "CCL20-CCR6 ligand-receptor interactions, with bubble size proportional to "
    "communication probability. Interactions are present in adjacent_LG, adjacent_HG, "
    "and tumor_LG but absent from normal and tumor_HG.")

add_figure(doc, f"{RES}/06_cellchat/comparative/bubble_CXCL_all_conditions.png", 23,
    "CXCL signaling pathway across all disease stages. Bubble plot showing significant "
    "CXCL ligand-receptor pairs across all five conditions. Bubble size represents "
    "communication probability; color indicates disease stage.")

add_figure(doc, f"{RES}/06_cellchat/comparative/bubble_MHC-II_all_conditions.png", 24,
    "MHC-II signaling pathway across disease stages. Bubble plot showing MHC-II "
    "ligand-receptor interactions. MHC-II signaling is absent from normal tissue and "
    "emerges in adjacent_HG and tumor conditions, reflecting increased antigen "
    "presentation by myeloid and dendritic cells.")

add_figure(doc, f"{RES}/06_cellchat/comparative/bubble_VEGF_all_conditions.png", 25,
    "VEGF signaling pathway across disease stages. Bubble plot showing VEGF "
    "ligand-receptor interactions. VEGF signaling is absent from normal tissue and "
    "progressively increases in adjacent and tumor conditions, reflecting the "
    "angiogenic switch in prostate cancer development.")

# Table 7: CCL20-CCR6
add_table_caption(doc, 7,
    "CCL20-CCR6 ligand-receptor interactions across disease stages. Table shows source "
    "cell type, target cell type, communication probability, p-value, and disease stage "
    "for all significant CCL20-CCR6 interactions.")

ccl20 = pd.read_csv(f"{RES}/06_cellchat/comparative/CCL20_CCR6_interactions.csv")
t7 = doc.add_table(rows=1, cols=6)
t7.style = 'Table Grid'
style_header_row(t7.rows[0])
for i, h in enumerate(["Source","Target","Ligand","Receptor","Probability","Condition"]):
    t7.rows[0].cells[i].text = h
    for para in t7.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(9); run.font.name = 'Times New Roman'

for idx, (_, row) in enumerate(ccl20.iterrows()):
    tr = t7.add_row()
    bg = "FFF2CC" if idx % 2 == 0 else "FFFDE7"
    vals = [row["source"], row["target"], row["ligand"], row["receptor"],
            f"{row['prob']:.2e}", cond_labels.get(row["condition"], row["condition"])]
    for i, val in enumerate(vals):
        tr.cells[i].text = str(val)
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER if i > 1 else WD_ALIGN_PARAGRAPH.LEFT
            for run in para.runs:
                run.font.size = Pt(9); run.font.name = 'Times New Roman'

set_cell_borders(t7)
doc.add_paragraph()
doc.add_page_break()
doc.save(OUT)
print("✓ Results 3.4.6 saved")


✓ Results 3.4.6 saved


Write Discussion section

In [27]:

# ── DISCUSSION ────────────────────────────────────────────────────────────────
add_heading(doc, "4. Discussion", 1)

add_body(doc,
    "This study presents a comprehensive single-cell transcriptomic analysis of the "
    "prostate cancer tumor microenvironment across five clinically defined disease stages, "
    "integrating cell type annotation, lineage trajectory inference, and cell-cell "
    "communication network analysis. Our findings reveal a progressive, multi-dimensional "
    "remodeling of the TME from normal prostate tissue through pre-malignant adjacent "
    "tissue to established low-grade and high-grade cancer, with distinct molecular "
    "mechanisms operating at each stage of disease progression."
)

add_body(doc,
    "Progressive TME Rewiring and Communication Complexity. One of the most striking "
    "findings of this study is the progressive increase in intercellular communication "
    "complexity across disease stages, with a 78% increase in significant L-R interactions "
    "and 108% increase in active signaling pathways from normal to low-grade tumor. "
    "Critically, this increase reflects the addition of new communication programs rather "
    "than replacement of existing ones — all 25 pathways active in normal tissue remained "
    "active in tumor tissue, with 32 additional pathways gained. This pattern of "
    "progressive accumulation rather than pathway switching suggests that tumor "
    "development involves the co-option of new signaling axes while maintaining "
    "homeostatic communication programs, consistent with the concept of TME 'education' "
    "rather than wholesale reprogramming [15]. The slight decrease in interaction count "
    "and pathway number in high-grade tumor relative to low-grade tumor may reflect "
    "progressive immune exclusion, T cell exhaustion, or reduced transcriptional "
    "diversity in the immunosuppressed high-grade TME — a phenomenon observed in other "
    "cancer types where advanced disease is associated with reduced immune infiltration [16]."
)

add_body(doc,
    "Validation of the CCL20-CCR6 Myeloid-Treg Recruitment Axis. Our analysis validated "
    "the CCL20-CCR6 myeloid-to-Treg recruitment axis identified by Mei et al. (2023) as "
    "a key immunosuppressive mechanism in prostate cancer [8]. CCL20 is a CC chemokine "
    "that binds exclusively to CCR6, which is expressed on Tregs, immature dendritic "
    "cells, and a subset of CD4+ T cells. Myeloid cell-derived CCL20 recruits CCR6+ "
    "Tregs to the tumor site, where they suppress anti-tumor immunity through multiple "
    "mechanisms including IL-10 and TGF-β secretion, CTLA-4-mediated co-stimulation "
    "blockade, and direct cytotoxic killing of effector T cells [17]. The restriction "
    "of CCL20-CCR6 signaling to adjacent and early tumor stages — with absence from "
    "normal tissue and high-grade tumor — suggests a temporal window of myeloid-driven "
    "Treg recruitment during early TME remodeling. In high-grade disease, alternative "
    "immunosuppressive mechanisms (such as HLA-E/NKG2A) may supersede CCL20-CCR6 as "
    "the dominant immune evasion strategy, potentially explaining the absence of this "
    "axis in tumor_HG."
)

add_body(doc,
    "HLA-E/NKG2A: A Novel Immune Checkpoint in High-Grade Prostate Cancer. Among the "
    "most clinically significant findings of this study is the identification of "
    "HLA-E/NKG2A inhibitory checkpoint signaling as a tumor_HG-specific immune evasion "
    "mechanism. HLA-E is a non-classical MHC-I molecule with limited polymorphism that "
    "presents peptides derived from the signal sequences of classical HLA-A, -B, and -C "
    "molecules. Under normal conditions, HLA-E expression on healthy cells signals "
    "'self' to NK cells via the activating NKG2C/CD94 receptor, preventing NK cell-"
    "mediated killing. In cancer, tumor cells and immunosuppressive myeloid cells "
    "upregulate HLA-E, which engages the inhibitory NKG2A/CD94 receptor on CD8+ CTL "
    "and NK cells, delivering a potent inhibitory signal that suppresses cytotoxic "
    "activity [18]. The multi-cellular nature of HLA-E/NKG2A signaling in our data — "
    "with interactions detected from CD4+ T cells, CD8+ CTL, Tregs, and Myeloid cells "
    "to CD8+ CTL — suggests a coordinated, network-level immune suppression program "
    "rather than a cell-autonomous mechanism. This finding has direct therapeutic "
    "implications: anti-NKG2A antibodies (such as monalizumab) are currently in clinical "
    "development for multiple cancer types and have shown promising results in combination "
    "with PD-L1 blockade [19]. Our data provide a mechanistic rationale for investigating "
    "NKG2A blockade in high-grade prostate cancer."
)

add_body(doc,
    "MHC-II Upregulation: Antigen Presentation in an Immunosuppressive Context. The "
    "emergence of MHC-II signaling in adjacent_HG and tumor conditions presents an "
    "apparent paradox: increased antigen presentation capacity alongside progressive "
    "immunosuppression. This paradox can be resolved by considering the cellular context "
    "of MHC-II signaling. In the prostate cancer TME, MHC-II is predominantly expressed "
    "by tumor-associated macrophages (TAMs) and dendritic cells that have been polarized "
    "toward an immunosuppressive (M2-like) phenotype. These cells present tumor antigens "
    "to CD4+ T cells in a tolerogenic context — lacking co-stimulatory signals and "
    "enriched in inhibitory ligands — preferentially inducing Treg differentiation rather "
    "than effector T helper cell activation [20]. Thus, increased MHC-II signaling in "
    "tumor tissue may paradoxically amplify immunosuppression by expanding the Treg pool "
    "through antigen-specific tolerance induction."
)

add_body(doc,
    "Angiogenic Switching and Stromal Remodeling. The progressive gain of VEGF, ANGPT, "
    "PDGF, and CDH5 signaling in tumor conditions reflects the angiogenic switch that "
    "accompanies tumor development. VEGF-A signaling through FLT1 and KDR on endothelial "
    "cells drives neovascularization, while ANGPT-Tie2 signaling regulates vessel "
    "maturation and pericyte recruitment. The co-emergence of VEGF and ANGPT signaling "
    "in tumor_LG suggests coordinated vascular remodeling from the earliest stages of "
    "tumor formation. Beyond its pro-angiogenic role, VEGF has well-established "
    "immunosuppressive effects, including impairment of dendritic cell maturation, "
    "promotion of Treg expansion, and induction of T cell exhaustion [21]. The "
    "identification of Fibroblasts and Pericytes as dominant VEGF senders in tumor "
    "conditions highlights the stromal contribution to tumor vascularization and "
    "immune evasion."
)

add_body(doc,
    "Prostaglandin-Mediated Immunosuppression. The emergence of Prostaglandin (PGE2) "
    "signaling in adjacent_LG and all subsequent stages represents an early and "
    "sustained immunosuppressive mechanism. PGE2, produced by cyclooxygenase-2 (COX-2) "
    "in tumor cells and myeloid cells, signals through PTGER4 (EP4 receptor) on T cells "
    "to suppress effector function, promote Treg differentiation, and inhibit NK cell "
    "activity [22]. The detection of PGE2-PTGES3 → PTGER4 interactions between T cell "
    "populations in tumor_HG — absent from normal tissue — suggests T cell-intrinsic "
    "prostaglandin signaling as a mechanism of T cell exhaustion in advanced disease. "
    "This finding supports the rationale for COX-2 inhibition as an immunomodulatory "
    "strategy in prostate cancer."
)

add_body(doc,
    "Epithelial Lineage Trajectory and Tumor Cell Origins. PAGA trajectory analysis "
    "revealed a continuous epithelial lineage axis from normal Epithelial cells through "
    "Basal epithelial cells to Tumor cells, with progressive pseudotime values of 0.09, "
    "0.27, and 0.43, respectively. This trajectory is consistent with the luminal "
    "epithelial origin of most prostate adenocarcinomas, with basal cells serving as "
    "an intermediate progenitor state [23]. The strong PAGA connectivity between Basal "
    "epithelial cells and Tumor cells (0.41) supports a model in which basal progenitor "
    "cells, rather than terminally differentiated luminal cells, are the primary cell "
    "of origin for prostate cancer — a hypothesis supported by experimental evidence "
    "from mouse models and human tissue studies [24]."
)

add_body(doc,
    "Limitations. Several limitations of this study should be acknowledged. First, the "
    "cross-sectional design of GSE181294 means that disease stage comparisons reflect "
    "between-patient rather than within-patient progression, and patient-specific "
    "variation may confound stage-specific findings. Second, CellChat communication "
    "probabilities are statistical estimates based on gene expression co-occurrence "
    "and do not constitute functional validation of ligand-receptor interactions; "
    "experimental validation (e.g., co-culture assays, spatial transcriptomics) is "
    "required to confirm the biological significance of identified interactions. Third, "
    "the DPT pseudotime analysis was rooted in normal epithelial cells, which is "
    "appropriate for the epithelial lineage but may not capture the developmental "
    "trajectories of immune and stromal populations. Fourth, the small number of "
    "tumor cells (153) in the dataset limited our ability to characterize the "
    "malignant epithelial compartment in detail. Future studies with larger tumor "
    "cell numbers and spatial transcriptomic validation will be needed to fully "
    "characterize the tumor cell-intrinsic communication programs."
)

add_body(doc,
    "Conclusions and Future Directions. This study provides a comprehensive, "
    "multi-module computational analysis of the prostate cancer TME, revealing "
    "progressive immune evasion, angiogenic switching, and stromal remodeling as "
    "disease advances from normal tissue to high-grade cancer. The identification "
    "of HLA-E/NKG2A inhibitory checkpoint signaling as a high-grade-specific immune "
    "evasion mechanism and the validation of the CCL20-CCR6 myeloid-Treg recruitment "
    "axis provide actionable therapeutic targets. Future directions include: (1) "
    "spatial transcriptomic validation of key L-R interactions using Visium or "
    "MERFISH platforms; (2) functional validation of HLA-E/NKG2A and CCL20-CCR6 "
    "axes using co-culture and in vivo models; (3) integration with clinical outcome "
    "data to identify communication signatures associated with treatment response and "
    "disease recurrence; and (4) extension of the analysis to include metastatic "
    "castration-resistant prostate cancer to map the full spectrum of TME evolution."
)

doc.add_page_break()
doc.save(OUT)
print("✓ Discussion saved")


✓ Discussion saved


Write References and Supplementary Tables

In [29]:

# ── REFERENCES ────────────────────────────────────────────────────────────────
add_heading(doc, "References", 1)

references = [
    "Sung H, Ferlay J, Siegel RL, et al. Global Cancer Statistics 2020: GLOBOCAN Estimates of Incidence and Mortality Worldwide for 36 Cancers in 185 Countries. CA Cancer J Clin. 2021;71(3):209-249.",
    "Binnewies M, Roberts EW, Kersten K, et al. Understanding the tumor immune microenvironment (TIME) for effective therapy. Nat Med. 2018;24(5):541-550.",
    "Sfanos KS, Yegnasubramanian S, Nelson WG, De Marzo AM. The inflammatory microenvironment and microbiome in prostate cancer development. Nat Rev Urol. 2018;15(1):11-24.",
    "Vitkin N, Nersesian S, Siemens DR, Koti M. The Tumor Immune Contexture of Prostate Cancer. Front Immunol. 2019;10:603.",
    "Papalexi E, Satija R. Single-cell RNA sequencing to explore immune cell heterogeneity. Nat Rev Immunol. 2018;18(1):35-45.",
    "Tuong ZK, Loudon KW, Berry B, et al. Resolving the immune landscape of human prostate at a single-cell level in health and cancer. Cell Rep. 2021;37(12):110132.",
    "Chen S, Zhu G, Yang Y, et al. Single-cell analysis reveals transcriptomic remodellings in distinct cell types that contribute to human prostate cancer progression. Nat Cell Biol. 2021;23(1):87-98.",
    "Mei W, Deng Y, Zhao S, et al. Dissecting the tumor microenvironment of prostate cancer at single-cell resolution. Front Oncol. 2023;12:1079926. PMID: 36750562.",
    "Hao Y, Hao S, Andersen-Nissen E, et al. Integrated analysis of multimodal single-cell data. Cell. 2021;184(13):3573-3587.e29.",
    "Korsunsky I, Millard N, Fan J, et al. Fast, sensitive and accurate integration of single-cell data with Harmony. Nat Methods. 2019;16(12):1289-1296.",
    "Wolf FA, Angerer P, Theis FJ. SCANPY: large-scale single-cell gene expression data analysis. Genome Biol. 2018;19(1):15.",
    "Wolf FA, Hamey FK, Plass M, et al. PAGA: graph abstraction reconciles clustering with trajectory inference through a topology preserving map of single cells. Genome Biol. 2019;20(1):59.",
    "Haghverdi L, Büttner M, Wolf FA, Buettner F, Theis FJ. Diffusion pseudotime robustly reconstructs lineage branching. Nat Methods. 2016;13(10):845-848.",
    "Jin S, Guerrero-Juarez CF, Zhang L, et al. Inference and analysis of cell-cell communication using CellChat. Nat Commun. 2021;12(1):1088.",
    "Quail DF, Joyce JA. Microenvironmental regulation of tumor progression and metastasis. Nat Med. 2013;19(11):1423-1437.",
    "Thommen DS, Schumacher TN. T Cell Dysfunction in Cancer. Cancer Cell. 2018;33(4):547-562.",
    "Plitas G, Rudensky AY. Regulatory T Cells: Differentiation and Function. Cancer Immunol Res. 2016;4(9):721-725.",
    "André P, Denis C, Soulas C, et al. Anti-NKG2A mAb Is a Checkpoint Inhibitor that Promotes Anti-tumor Immunity by Unleashing Both T and NK Cells. Cell. 2018;175(7):1731-1743.e13.",
    "van Montfoort N, Borst L, Korrer MJ, et al. NKG2A Blockade Potentiates CD8 T Cell Immunity Induced by Cancer Vaccines. Cell. 2018;175(7):1744-1755.e15.",
    "Wculek SK, Cueto FJ, Mujal AM, Melero I, Krummel MF, Sancho D. Dendritic cells in cancer immunology and immunotherapy. Nat Rev Immunol. 2020;20(1):7-24.",
    "Voron T, Colussi O, Marcheteau E, et al. VEGF-A modulates expression of inhibitory checkpoints on CD8+ T cells in tumors. J Exp Med. 2015;212(2):139-148.",
    "Zelenay S, van der Veen AG, Böttcher JP, et al. Cyclooxygenase-Dependent Tumor Growth through Evasion of Immunity. Cell. 2015;162(6):1257-1270.",
    "Shen MM, Abate-Shen C. Molecular genetics of prostate cancer: new prospects for old challenges. Genes Dev. 2010;24(18):1967-2000.",
    "Wang X, Kruithof-de Julio M, Economides KD, et al. A luminal epithelial stem cell that is a cell of origin for prostate cancer. Nature. 2009;461(7263):495-500.",
]

for i, ref in enumerate(references):
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(4)
    p.paragraph_format.left_indent = Cm(0.8)
    p.paragraph_format.first_line_indent = Cm(-0.8)
    r1 = p.add_run(f"[{i+1}] ")
    r1.bold = True; r1.font.size = Pt(10); r1.font.name = 'Times New Roman'
    r2 = p.add_run(ref)
    r2.font.size = Pt(10); r2.font.name = 'Times New Roman'

doc.add_page_break()

# ── SUPPLEMENTARY TABLES ──────────────────────────────────────────────────────
add_heading(doc, "Supplementary Tables", 1)

# Supp Table 1: Interaction count summary per condition
add_heading(doc, "Supplementary Table 1: Significant Interaction Summary per Condition", 2)
add_body(doc,
    "The following table summarizes the total number of significant ligand-receptor "
    "interactions per condition, as inferred by CellChat v2. Full interaction tables "
    "(including source, target, ligand, receptor, probability, and pathway) are provided "
    "as separate CSV files."
)

supp_data = [
    ["Normal", "964", "25", "7,095 (all conditions combined)"],
    ["Adjacent LG", "1,263", "35", ""],
    ["Adjacent HG", "1,522", "52", ""],
    ["Tumor LG", "1,716", "52", ""],
    ["Tumor HG", "1,630", "46", ""],
]
ts1 = doc.add_table(rows=1, cols=4)
ts1.style = 'Table Grid'
style_header_row(ts1.rows[0])
for i, h in enumerate(["Condition","Interactions (n)","Pathways (n)","Note"]):
    ts1.rows[0].cells[i].text = h
    for para in ts1.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(10); run.font.name = 'Times New Roman'
for idx, vals in enumerate(supp_data):
    tr = ts1.add_row()
    bg = "F2F7FF" if idx % 2 == 0 else "FFFFFF"
    for i, val in enumerate(vals):
        tr.cells[i].text = val
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in para.runs:
                run.font.size = Pt(10); run.font.name = 'Times New Roman'
set_cell_borders(ts1)
doc.add_paragraph()

# Supp Table 2: PAGA connectivity (top 20)
add_heading(doc, "Supplementary Table 2: PAGA Connectivity Matrix (Top 20 Connections)", 2)
add_body(doc,
    "Top 20 PAGA connectivity values between cell type pairs, sorted by connectivity "
    "strength. Only connections with connectivity ≥ 0.05 are shown."
)
ts2 = doc.add_table(rows=1, cols=3)
ts2.style = 'Table Grid'
style_header_row(ts2.rows[0])
for i, h in enumerate(["Cell Type 1","Cell Type 2","PAGA Connectivity"]):
    ts2.rows[0].cells[i].text = h
    for para in ts2.rows[0].cells[i].paragraphs:
        para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        for run in para.runs:
            run.bold = True; run.font.color.rgb = RGBColor(255,255,255)
            run.font.size = Pt(10); run.font.name = 'Times New Roman'
for idx, (_, row) in enumerate(pairs_df.iterrows()):
    tr = ts2.add_row()
    bg = "F2F7FF" if idx % 2 == 0 else "FFFFFF"
    for i, val in enumerate([row["Cell Type 1"], row["Cell Type 2"], f"{row['Connectivity']:.4f}"]):
        tr.cells[i].text = str(val)
        set_cell_bg(tr.cells[i], bg)
        for para in tr.cells[i].paragraphs:
            para.alignment = WD_ALIGN_PARAGRAPH.CENTER
            for run in para.runs:
                run.font.size = Pt(10); run.font.name = 'Times New Roman'
set_cell_borders(ts2)
doc.add_paragraph()

doc.save(OUT)
print("✓ References and Supplementary Tables saved")


✓ References and Supplementary Tables saved


Reload document and re-establish kernel state

In [31]:

# Re-establish kernel state after compaction
# All imports and helpers need to be re-declared since kernel was reset

import sys
sys.path.insert(0, "/mnt/shared-workspace/pip-packages")

from docx import Document
from docx.shared import Pt, Cm, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
import copy, os, pandas as pd

OUT = "/workspace/report_GSE181294_TME_analysis.docx"
doc = Document(OUT)

print(f"Document loaded: {len(doc.paragraphs)} paragraphs, {len(doc.tables)} tables")
print(f"Sections: {len(doc.sections)}")


Document loaded: 190 paragraphs, 10 tables
Sections: 1


Apply final formatting: margins, page numbers, line spacing

In [33]:

# ── FINAL FORMATTING POLISH ───────────────────────────────────────────────────
# 1. Set document-wide default font and line spacing
# 2. Add page numbers (bottom center)
# 3. Ensure all body paragraphs have 1.5 line spacing
# 4. Ensure margins are 2.5 cm all sides

from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

# ── 1. Margins ────────────────────────────────────────────────────────────────
section = doc.sections[0]
section.top_margin    = Cm(2.5)
section.bottom_margin = Cm(2.5)
section.left_margin   = Cm(2.5)
section.right_margin  = Cm(2.5)

# ── 2. Page numbers (bottom center) ──────────────────────────────────────────
def add_page_numbers(section):
    footer = section.footer
    # Clear existing footer content
    for para in footer.paragraphs:
        for run in para.runs:
            run.text = ""
    
    if not footer.paragraphs:
        footer_para = footer.add_paragraph()
    else:
        footer_para = footer.paragraphs[0]
    
    footer_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add page number field
    run = footer_para.add_run()
    fldChar1 = OxmlElement('w:fldChar')
    fldChar1.set(qn('w:fldCharType'), 'begin')
    run._r.append(fldChar1)
    
    instrText = OxmlElement('w:instrText')
    instrText.set(qn('xml:space'), 'preserve')
    instrText.text = 'PAGE'
    run._r.append(instrText)
    
    fldChar2 = OxmlElement('w:fldChar')
    fldChar2.set(qn('w:fldCharType'), 'end')
    run._r.append(fldChar2)
    
    run.font.size = Pt(10)
    run.font.name = 'Times New Roman'

add_page_numbers(section)

# ── 3. Line spacing: apply 1.5 to all body paragraphs ────────────────────────
from docx.shared import Pt
from docx.oxml.ns import qn

HEADING_STYLES = {'Heading 1', 'Heading 2', 'Heading 3', 'Title', 'Subtitle'}

for para in doc.paragraphs:
    style_name = para.style.name if para.style else ""
    # Skip headings, captions, table content
    if any(h in style_name for h in HEADING_STYLES):
        continue
    if not para.text.strip():
        continue
    
    pPr = para._p.get_or_add_pPr()
    spacing = pPr.find(qn('w:spacing'))
    if spacing is None:
        spacing = OxmlElement('w:spacing')
        pPr.append(spacing)
    spacing.set(qn('w:line'), '360')       # 360 twips = 1.5 × 240 (single)
    spacing.set(qn('w:lineRule'), 'auto')
    spacing.set(qn('w:after'), '120')      # 6pt after paragraph

# ── 4. Default document font (Normal style) ───────────────────────────────────
from docx.oxml.ns import qn as _qn
normal_style = doc.styles['Normal']
normal_style.font.name = 'Times New Roman'
normal_style.font.size = Pt(12)

doc.save(OUT)
print("✓ Formatting polish applied: margins, page numbers, line spacing, default font")


✓ Formatting polish applied: margins, page numbers, line spacing, default font


Verify document structure and element counts

In [35]:

# ── FINAL VERIFICATION ────────────────────────────────────────────────────────
import os

file_size = os.path.getsize(OUT)
print(f"File size: {file_size / 1024 / 1024:.2f} MB")

# Count document elements
n_paragraphs = len(doc.paragraphs)
n_tables = len(doc.tables)

# Count embedded images
from docx.opc.constants import RELATIONSHIP_TYPE as RT
n_images = len([r for r in doc.part.rels.values() if "image" in r.reltype])

# Count headings
headings = [p for p in doc.paragraphs if p.style.name.startswith('Heading')]
h1 = [p for p in headings if p.style.name == 'Heading 1']
h2 = [p for p in headings if p.style.name == 'Heading 2']

print(f"\nDocument structure:")
print(f"  Paragraphs: {n_paragraphs}")
print(f"  Tables: {n_tables}")
print(f"  Embedded images: {n_images}")
print(f"  H1 sections: {len(h1)}")
print(f"  H2 subsections: {len(h2)}")

print(f"\nH1 sections:")
for h in h1:
    print(f"  • {h.text}")

print(f"\nH2 subsections:")
for h in h2:
    print(f"  • {h.text}")


File size: 4.74 MB

Document structure:
  Paragraphs: 190
  Tables: 10
  Embedded images: 24
  H1 sections: 8
  H2 subsections: 14

H1 sections:
  • Abstract
  • 1. Introduction
  • 2. Materials and Methods
  • 3. Results
  • 3. Results
  • 4. Discussion
  • References
  • Supplementary Tables

H2 subsections:
  • 2.1 Dataset and Study Design
  • 2.2 Quality Control and Cell Filtering
  • 2.3 Normalization, Feature Selection, and Dimensionality Reduction
  • 2.4 Clustering and Cell Type Annotation
  • 2.5 Trajectory Inference: PAGA and Diffusion Pseudotime
  • 2.6 Cell-Cell Communication Analysis: CellChat v2
  • 2.7 Statistical Analysis and Visualization
  • 3.1 Quality Control and Dataset Overview
  • 3.1 Quality Control and Dataset Overview
  • 3.2 Single-Cell Transcriptomic Landscape of the Prostate TME
  • 3.3 Lineage Trajectory and Pseudotime Analysis
  • 3.4 Cell-Cell Communication Networks Across Disease Progression
  • Supplementary Table 1: Significant Interaction Summary per

Inspect heading positions to identify duplicates

In [37]:

# Fix duplicate headings: remove the duplicate "3. Results" and "3.1 Quality Control" 
# These appear to be from a prior partial write that got duplicated

# Strategy: find and remove duplicate heading paragraphs
from docx.oxml.ns import qn

def remove_paragraph(para):
    """Remove a paragraph from the document."""
    p = para._element
    p.getparent().remove(p)

# Find all H1 and H2 headings with their positions
all_paras = list(doc.paragraphs)
heading_positions = []
for i, para in enumerate(all_paras):
    if para.style.name in ('Heading 1', 'Heading 2'):
        heading_positions.append((i, para.style.name, para.text))

print("All headings with positions:")
for pos, style, text in heading_positions:
    print(f"  [{pos:3d}] {style}: {text}")


All headings with positions:
  [  5] Heading 1: Abstract
  [ 12] Heading 1: 1. Introduction
  [ 19] Heading 1: 2. Materials and Methods
  [ 20] Heading 2: 2.1 Dataset and Study Design
  [ 22] Heading 2: 2.2 Quality Control and Cell Filtering
  [ 24] Heading 2: 2.3 Normalization, Feature Selection, and Dimensionality Reduction
  [ 26] Heading 2: 2.4 Clustering and Cell Type Annotation
  [ 28] Heading 2: 2.5 Trajectory Inference: PAGA and Diffusion Pseudotime
  [ 30] Heading 2: 2.6 Cell-Cell Communication Analysis: CellChat v2
  [ 32] Heading 2: 2.7 Statistical Analysis and Visualization
  [ 35] Heading 1: 3. Results
  [ 36] Heading 2: 3.1 Quality Control and Dataset Overview
  [ 39] Heading 1: 3. Results
  [ 40] Heading 2: 3.1 Quality Control and Dataset Overview
  [ 49] Heading 2: 3.2 Single-Cell Transcriptomic Landscape of the Prostate TME
  [ 63] Heading 2: 3.3 Lineage Trajectory and Pseudotime Analysis
  [ 80] Heading 2: 3.4 Cell-Cell Communication Networks Across Disease Progressio

Inspect duplicate block content

In [39]:

# Remove the duplicate "3. Results" (pos 39) and "3.1 Quality Control" (pos 40)
# Also need to remove any content between pos 39-48 that duplicates pos 35-48

# Get fresh paragraph list
all_paras = list(doc.paragraphs)

# The duplicate block starts at index 39 ("3. Results") and 40 ("3.1 Quality Control...")
# We need to check what's between pos 40 and pos 49 (3.2 heading)
print("Content between duplicate 3.1 and 3.2:")
for i in range(39, 49):
    para = all_paras[i]
    text_preview = para.text[:80] if para.text else "[empty]"
    print(f"  [{i}] {para.style.name}: {text_preview}")


Content between duplicate 3.1 and 3.2:
  [39] Heading 1: 3. Results
  [40] Heading 2: 3.1 Quality Control and Dataset Overview
  [41] Normal: After applying quality control filters (nFeature_RNA: 200–6,000; nCount_RNA: 500
  [42] Normal: Table 1. Quality control summary per sample. Columns show sample identifier, dis
  [43] Normal: [empty]
  [44] Normal: [empty]
  [45] Normal: Figure 1. Quality control metrics by disease stage. Violin plots showing the dis
  [46] Normal: [empty]
  [47] Normal: Figure 2. Cell counts per sample after quality control filtering. Bar chart show
  [48] Normal: [empty]


Check which duplicate block has images

In [41]:

# Check what's in the FIRST "3. Results" block (pos 35-38) to understand which is the real one
all_paras = list(doc.paragraphs)
print("First '3. Results' block (pos 35-38):")
for i in range(35, 39):
    para = all_paras[i]
    text_preview = para.text[:100] if para.text else "[empty]"
    print(f"  [{i}] {para.style.name}: {text_preview}")

# Check if there are images in the first block vs second block
# Images are in inline shapes, not paragraphs directly
# Let's check the XML around pos 35-48 for picture elements
from lxml import etree

print("\nChecking for images in first block (pos 35-38):")
for i in range(35, 39):
    para = all_paras[i]
    has_pic = para._p.find('.//' + qn('a:blip')) is not None or \
              para._p.find('.//' + qn('pic:pic')) is not None or \
              para._p.find('.//' + qn('w:drawing')) is not None
    print(f"  [{i}] has_image={has_pic}: {para.text[:60] if para.text else '[empty]'}")

print("\nChecking for images in second block (pos 39-48):")
for i in range(39, 49):
    para = all_paras[i]
    has_pic = para._p.find('.//' + qn('w:drawing')) is not None
    print(f"  [{i}] has_image={has_pic}: {para.text[:60] if para.text else '[empty]'}")


First '3. Results' block (pos 35-38):
  [35] Heading 1: 3. Results
  [36] Heading 2: 3.1 Quality Control and Dataset Overview
  [37] Normal: After applying quality control filters (nFeature_RNA: 200–6,000; nCount_RNA: 500–30,000; percent.mt 
  [38] Normal: Table 1. Quality control summary per sample. Columns show sample identifier, disease stage, patient 

Checking for images in first block (pos 35-38):
  [35] has_image=False: 3. Results
  [36] has_image=False: 3.1 Quality Control and Dataset Overview
  [37] has_image=False: After applying quality control filters (nFeature_RNA: 200–6,
  [38] has_image=False: Table 1. Quality control summary per sample. Columns show sa

Checking for images in second block (pos 39-48):
  [39] has_image=False: 3. Results
  [40] has_image=False: 3.1 Quality Control and Dataset Overview
  [41] has_image=False: After applying quality control filters (nFeature_RNA: 200–6,
  [42] has_image=False: Table 1. Quality control summary per sample. Columns show sa
  [

Find body child indices for duplicate removal

In [43]:

# The SECOND block (pos 39-48) has the images — it's the real one.
# The FIRST block (pos 35-38) is the incomplete duplicate — remove it.

# Also check if the first block has a table
print("Checking for tables near first block:")
# Tables are separate from paragraphs in python-docx
# Let's look at the raw XML body children to understand the order
body = doc.element.body
children = list(body)
print(f"Total body children: {len(children)}")

# Find the positions of our duplicate paragraphs in the body children
para_35 = all_paras[35]._p
para_39 = all_paras[39]._p

idx_35 = list(body).index(para_35)
idx_39 = list(body).index(para_39)
print(f"Body child index of para[35]: {idx_35}")
print(f"Body child index of para[39]: {idx_39}")

# Show body children between idx_35 and idx_39
print(f"\nBody children from {idx_35} to {idx_39}:")
for i in range(idx_35, idx_39 + 1):
    child = children[i]
    tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
    # Get text preview
    text = ''.join(child.itertext())[:60]
    print(f"  [{i}] <{tag}>: {text}")


Checking for tables near first block:
Total body children: 201
Body child index of para[35]: 35
Body child index of para[39]: 40

Body children from 35 to 40:
  [35] <p>: 3. Results3. Results3. Results
  [36] <p>: 3.1 Quality Control and Dataset Overview3.1 Quality Control 
  [37] <p>: After applying quality control filters (nFeature_RNA: 200–6,
  [38] <p>: Table 1. Quality control summary per sample. Columns show sa
  [39] <tbl>: 
  [40] <p>: 3. Results3. Results3. Results


Check second block for table presence

In [45]:

# The first block (body children 35-39) contains:
# [35] "3. Results" heading
# [36] "3.1 QC" heading  
# [37] body text
# [38] "Table 1." caption
# [39] <tbl> — the actual Table 1 data table
# [40] "3. Results" heading (second, real block with images)

# The second block (starting at body child 40) has the images.
# We need to KEEP the table from the first block but remove the duplicate headings/text.
# Actually, let's check if the second block also has a table...

body = doc.element.body
children = list(body)

# Find the next table after body child 40
print("Body children from 40 to 60:")
for i in range(40, 61):
    child = children[i]
    tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
    text = ''.join(child.itertext())[:60]
    print(f"  [{i}] <{tag}>: {text}")


Body children from 40 to 60:
  [40] <p>: 3. Results3. Results3. Results
  [41] <p>: 3.1 Quality Control and Dataset Overview3.1 Quality Control 
  [42] <p>: After applying quality control filters (nFeature_RNA: 200–6,
  [43] <p>: Table 1. Quality control summary per sample. Columns show sa
  [44] <tbl>: Sample IDSample IDSample IDConditionConditionConditionPatien
  [45] <p>: 
  [46] <p>: 
  [47] <p>: Figure 1. Quality control metrics by disease stage. Violin p
  [48] <p>: 
  [49] <p>: Figure 2. Cell counts per sample after quality control filte
  [50] <p>: 
  [51] <p>: 3.2 Single-Cell Transcriptomic Landscape of the Prostate TME
  [52] <p>: Unsupervised clustering of 149,772 cells at resolution 0.6 y
  [53] <p>: Analysis of cell type proportions across disease stages reve
  [54] <p>: 
  [55] <p>: Figure 3. UMAP visualization of 149,772 cells colored by clu
  [56] <p>: 
  [57] <p>: Figure 4. UMAP visualization colored by disease stage. Cells
  [58] <p>: 
  [59] <p>: Figure 5. UMAP visua

Remove duplicate Results/3.1 block from document

In [47]:

# Perfect. The second block (40-50) has BOTH the table AND the images.
# The first block (35-39) is the incomplete duplicate — remove body children 35-39.

# Remove body children 35-39 (inclusive): 5 elements
# Must remove in reverse order to preserve indices
body = doc.element.body
children = list(body)

# Elements to remove: indices 35, 36, 37, 38, 39
to_remove = [children[i] for i in range(35, 40)]  # 35,36,37,38,39

for elem in to_remove:
    body.remove(elem)

# Verify
children_after = list(body)
print(f"Body children after removal: {len(children_after)} (was 201, removed 5)")

# Check the area around where the duplicate was
print("\nBody children 33-45 after removal:")
for i in range(33, 46):
    child = children_after[i]
    tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
    text = ''.join(child.itertext())[:70]
    print(f"  [{i}] <{tag}>: {text}")

doc.save(OUT)
print("\n✓ Duplicate section removed and document saved")


Body children after removal: 196 (was 201, removed 5)

Body children 33-45 after removal:
  [33] <p>: Cell type proportions were computed as the fraction of cells of each t
  [34] <p>: 
  [35] <p>: 3. Results3. Results3. Results
  [36] <p>: 3.1 Quality Control and Dataset Overview3.1 Quality Control and Datase
  [37] <p>: After applying quality control filters (nFeature_RNA: 200–6,000; nCoun
  [38] <p>: Table 1. Quality control summary per sample. Columns show sample ident
  [39] <tbl>: Sample IDSample IDSample IDConditionConditionConditionPatientPatientPa
  [40] <p>: 
  [41] <p>: 
  [42] <p>: Figure 1. Quality control metrics by disease stage. Violin plots showi
  [43] <p>: 
  [44] <p>: Figure 2. Cell counts per sample after quality control filtering. Bar 
  [45] <p>: 

✓ Duplicate section removed and document saved


Check raw XML of Results heading

In [49]:

# The heading text shows "3. Results3. Results3. Results" — this is a rendering artifact
# from how python-docx reads back styled paragraphs. Let's verify the actual XML text.

body = doc.element.body
children = list(body)

# Check the actual XML of the "3. Results" heading
heading_elem = children[35]
from lxml import etree
print("Raw XML of heading at [35]:")
print(etree.tostring(heading_elem, pretty_print=True).decode()[:500])


Raw XML of heading at [35]:
<w:p xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main" xmlns:wpc="http://schemas.microsoft.com/office/word/2010/wordprocessingCanvas" xmlns:mo="http://schemas.microsoft.com/office/mac/office/2008/main" xmlns:mc="http://schemas.openxmlformats.org/markup-compatibility/2006" xmlns:mv="urn:schemas-microsoft-com:mac:vml" xmlns:o="urn:schemas-microsoft-com:office:office" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" xmlns:m="http://schemas.openxmlfo


Check text runs in heading element

In [51]:

# Get just the text runs from the heading
from docx.oxml.ns import qn

heading_elem = children[35]
runs = heading_elem.findall('.//' + qn('w:r'))
print(f"Number of runs in heading: {len(runs)}")
for i, run in enumerate(runs):
    t = run.find(qn('w:t'))
    if t is not None:
        print(f"  Run {i}: '{t.text}'")


Number of runs in heading: 1
  Run 0: '3. Results'


Final heading structure verification

In [53]:

# Good — the heading has only 1 run with correct text "3. Results"
# The repeated text in itertext() was a display artifact from the style definition.
# The document is clean.

# Final verification: re-read and count all headings
doc2 = Document(OUT)
headings = [(p.style.name, p.text) for p in doc2.paragraphs if p.style.name.startswith('Heading')]
print("Final heading structure:")
for style, text in headings:
    indent = "  " if style == "Heading 2" else ""
    print(f"  {indent}[{style}] {text}")

print(f"\nTotal: {len([h for h in headings if h[0]=='Heading 1'])} H1, "
      f"{len([h for h in headings if h[0]=='Heading 2'])} H2")

import os
print(f"\nFinal file size: {os.path.getsize(OUT)/1024/1024:.2f} MB")


Final heading structure:
  [Heading 1] Abstract
  [Heading 1] 1. Introduction
  [Heading 1] 2. Materials and Methods
    [Heading 2] 2.1 Dataset and Study Design
    [Heading 2] 2.2 Quality Control and Cell Filtering
    [Heading 2] 2.3 Normalization, Feature Selection, and Dimensionality Reduction
    [Heading 2] 2.4 Clustering and Cell Type Annotation
    [Heading 2] 2.5 Trajectory Inference: PAGA and Diffusion Pseudotime
    [Heading 2] 2.6 Cell-Cell Communication Analysis: CellChat v2
    [Heading 2] 2.7 Statistical Analysis and Visualization
  [Heading 1] 3. Results
    [Heading 2] 3.1 Quality Control and Dataset Overview
    [Heading 2] 3.2 Single-Cell Transcriptomic Landscape of the Prostate TME
    [Heading 2] 3.3 Lineage Trajectory and Pseudotime Analysis
    [Heading 2] 3.4 Cell-Cell Communication Networks Across Disease Progression
  [Heading 3] 3.4.1 Global Communication Landscape
  [Heading 3] 3.4.2 Pathway Activity Across Disease Stages
  [Heading 3] 3.4.3 Tumor-Gained Si

Copy final document to /mnt/results/

In [55]:

# Perfect structure. Now copy to /mnt/results/
import shutil, subprocess, os

dest = "/mnt/results/report_GSE181294_TME_analysis.docx"

# Use shell cp (not shutil.copy) to avoid FUSE 0-byte issue on /mnt/results
result = subprocess.run(
    ["cp", OUT, dest],
    capture_output=True, text=True
)

if result.returncode == 0:
    size = os.path.getsize(dest)
    print(f"✓ Copied to /mnt/results/")
    print(f"  Destination size: {size/1024/1024:.2f} MB")
else:
    print(f"✗ Copy failed: {result.stderr}")


✓ Copied to /mnt/results/
  Destination size: 4.74 MB


Final content audit of the completed document

In [57]:

# Final content audit — count figures, tables, references
doc3 = Document("/mnt/results/report_GSE181294_TME_analysis.docx")

# Count figure captions
fig_captions = [p.text for p in doc3.paragraphs if p.text.startswith("Figure ")]
table_captions = [p.text for p in doc3.paragraphs if p.text.startswith("Table ") or p.text.startswith("Supplementary Table")]
ref_paras = [p.text for p in doc3.paragraphs if p.text.startswith("[") and p.text[1:3].strip().isdigit()]

# Count embedded images
from docx.opc.constants import RELATIONSHIP_TYPE as RT
n_images = len([r for r in doc3.part.rels.values() if "image" in r.reltype])

print("=== FINAL DOCUMENT AUDIT ===")
print(f"File: report_GSE181294_TME_analysis.docx ({os.path.getsize('/mnt/results/report_GSE181294_TME_analysis.docx')/1024/1024:.2f} MB)")
print(f"\nEmbedded images: {n_images}")
print(f"Tables: {len(doc3.tables)}")
print(f"Figure captions: {len(fig_captions)}")
print(f"Table captions: {len(table_captions)}")
print(f"References: {len(ref_paras)}")

print(f"\nFigure captions:")
for c in fig_captions:
    print(f"  • {c[:80]}")

print(f"\nTable captions:")
for c in table_captions:
    print(f"  • {c[:80]}")


=== FINAL DOCUMENT AUDIT ===
File: report_GSE181294_TME_analysis.docx (4.74 MB)

Embedded images: 24
Tables: 9
Figure captions: 25
Table captions: 10
References: 15

Figure captions:
  • Figure 1. Quality control metrics by disease stage. Violin plots showing the dis
  • Figure 2. Cell counts per sample after quality control filtering. Bar chart show
  • Figure 3. UMAP visualization of 149,772 cells colored by cluster identity. Leide
  • Figure 4. UMAP visualization colored by disease stage. Cells from all five condi
  • Figure 5. UMAP visualization colored by annotated cell type. Sixteen major cell 
  • Figure 6. Cell type composition across disease stages. Stacked bar chart showing
  • Figure 7. PAGA connectivity graph colored by cell type. Node size is proportiona
  • Figure 8. PAGA connectivity heatmap. Color intensity represents the connectivity
  • Figure 9. UMAP visualization colored by Diffusion Pseudotime (DPT). Pseudotime w
  • Figure 10. Pseudotime distribution by disease st